In [1]:
import boto3

In [1]:
[i for i in range(128, 3009, 128)]
print(len([i for i in range(10, 1000, 100)]))
{
  "lambdaARN": "arn:aws:lambda:ap-southeast-2:030103857128:function:workbench-matmul",
  "num": 3,
  "sla": {
    "value": "10"
  },
  "powerValues": [
    128,
    256,
    384,
    512,
    640,
    768,
    896,
    1024,
    1152,
    1280,
    1408,
    1536,
    1664,
    1792,
    1920,
    2048,
    2176,
    2304,
    2432,
    2560,
    2688,
    2816,
    2944,
    3008
  ],
  "payload": {
    "min": 10,
    "max": 10000,
    "stepSize": 100
  }
}

10


{'lambdaARN': 'arn:aws:lambda:ap-southeast-2:030103857128:function:workbench-matmul',
 'num': 3,
 'sla': {'value': '10'},
 'powerValues': [128,
  256,
  384,
  512,
  640,
  768,
  896,
  1024,
  1152,
  1280,
  1408,
  1536,
  1664,
  1792,
  1920,
  2048,
  2176,
  2304,
  2432,
  2560,
  2688,
  2816,
  2944,
  3008],
 'payload': {'min': 10, 'max': 10000, 'stepSize': 100}}

In [23]:
client = boto3.client('pricing', region_name='us-east-1')
res = client.get_attribute_values(
ServiceCode='AWSLambda',
AttributeName='productFamily',
MaxResults=12
)
print(res)

{'AttributeValues': [{'Value': 'Serverless'}], 'ResponseMetadata': {'RequestId': '7568d382-bc64-4f76-a00a-34165a1902aa', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Thu, 18 Jul 2024 06:06:59 GMT', 'content-type': 'application/x-amz-json-1.1', 'content-length': '44', 'connection': 'keep-alive', 'x-amzn-requestid': '7568d382-bc64-4f76-a00a-34165a1902aa'}, 'RetryAttempts': 0}}


In [16]:
response = client.describe_services()
# print(response)
for item in response['Services']:
    # print(item['ServiceCode'])
    if item['ServiceCode'] == 'AWSLambda':
        print(item)

             

{'ServiceCode': 'AWSLambda', 'AttributeNames': ['productFamily', 'termType', 'usagetype', 'locationType', 'Restriction', 'regionCode', 'servicecode', 'groupDescription', 'location', 'servicename', 'group']}


In [19]:
import boto3
import json

# Create a CloudWatch Logs client
client = boto3.client('logs', region_name='ap-southeast-2')

response = client.describe_log_groups()
log_group_names = []
log_groups = response['logGroups']
for log_group in log_groups:
    # print(log_group['logGroupName'])
    log_group_names.append(log_group['logGroupName'])

for log_group_name in log_group_names[:4]:
# Define the log group and log stream names
    print("processing for log group: ", log_group_name)
    # log_group_name = '/aws/lambda/testNewFunctionality'
    response = client.describe_log_streams(
        logGroupName=log_group_name,
        orderBy='LastEventTime',
        descending=True
    )
    log_streams = []
    for stream in response['logStreams']:
        log_streams.append(stream['logStreamName'])


    for log_stream_name in log_streams:
        # Get the log events from the log group and log stream
        response = client.get_log_events(
            logGroupName=log_group_name,
            logStreamName=log_stream_name
        )

        next_token = None
        i = 0
        while True:
            if next_token:
                response = client.get_log_events(
                    logGroupName=log_group_name,
                    logStreamName=log_stream_name,
                    startFromHead=True,
                    nextToken=next_token
                )
            else:
                response = client.get_log_events(
                    logGroupName=log_group_name,
                    logStreamName=log_stream_name,
                    startFromHead=True
                )
            
            if response['events'] == []:
                next_token = None
                break
            # Process the log events
            for event in response['events']:
                # Parse the JSON message
                try:
                    log_message = json.loads(event['message'])
                    print(log_message['memory_utilization'])
                except:
                    print("here is the incorrect log")
                    # print(event['message'])
            
            # Check if there are more log events available
            if 'nextForwardToken' in response:
                print("this is the next token: ", response['nextForwardToken'], 'for the iteration: ', i)
                next_token = response['nextForwardToken']
            else:
                break


            
            # Access the desired JSON keys and print them
            
        



processing for log group:  /aws/lambda-insights
5
5
this is the next token:  f/38382061801946249127119523376274852203443054967352328192/s for the iteration:  0
100
this is the next token:  f/38382056690682351859496290928694899422061221241588744192/s for the iteration:  0
60
this is the next token:  f/38379989614494627886962567852855389504334229111722868736/s for the iteration:  0
60
this is the next token:  f/38379988048335593339355435161070753590191283221296840704/s for the iteration:  0
60
this is the next token:  f/38379985305990655530846176674575581318774917966980579328/s for the iteration:  0


KeyboardInterrupt: 

#### Cloudwatch Logs Processing

In [25]:
import boto3
import utils as ut

lambda_arn = "arn:aws:lambda:ap-southeast-2:030103857128:function:testNewFunctionality"

# TODO: Fetch the last streamed log request id

# Create a CloudWatch Logs client
client = boto3.client('logs', region_name='ap-southeast-2')


# Get the list of log groups
log_groups = client.describe_log_groups()
log_group_names = [group['logGroupName'] for group 
                    in log_groups['logGroups'] 
                    if group['logGroupName'].endswith('/aws/lambda/' \
                                                        + lambda_arn.split(':')[-1])]

# Get the list of log streams
log_streams = []
for log_group_name in log_group_names:
    log_streams += client.describe_log_streams(logGroupName=log_group_name,
                                                orderBy='LastEventTime',
                                                descending=True)['logStreams']

# Get the list of log events
log_events = []
for log_stream in log_streams:
    response = client.get_log_events(
        logGroupName=log_group_name,
        logStreamName=log_stream['logStreamName']
    )
    log_events += response['events']
    


In [33]:
import re

def extract_duration(log):
    regex = r'Billed Duration: (\d+) ms'
    match = re.search(regex, log)
    if match:
        return int(match.group(1))
    return None

def extract_init_duration(log):
    regex = r'Init Duration: (\d+)'
    match = re.search(regex, log)
    if match:
        return int(match.group(1))
    return None

def extract_memory_size(log):
    regex = r'Memory Size: (\d+) MB'
    match = re.search(regex, log)
    if match:
        return int(match.group(1))
    return None

def extract_memory_used(log):
    regex = r'Memory Used: (\d+) MB'
    match = re.search(regex, log)
    if match:
        return int(match.group(1))
    return None


def extract_request_id(log):
    regex = r'^REPORT RequestId:\s+([a-f0-9-]+)'
    match = re.search(regex, log)
    if match:
        return match.group(1)
    return None

for log in log_events:
    try:
        # print(log['message'])
        match = re.search(r'^REPORT RequestId:', log['message'])
        # print(match)
        if match is not None:
            # print("here is the log message: ", log['message'])
            res = {
                'duration': extract_duration(log['message']),
                'init_duration': extract_init_duration(log['message']),
                'memory_size': extract_memory_size(log['message']),
                'memory_used': extract_memory_used(log['message']),
                'request_id': extract_request_id(log['message'])
            }
            print(res)
    except:
        continue



{'duration': 118, 'init_duration': 1191, 'memory_size': 128, 'memory_used': 118, 'request_id': 'cd353b77-ed0e-4fd5-8216-57803cf73389'}


#### Lambda Insights Processing

In [2]:
import boto3
import json
import time

# Create a CloudWatch Logs client
client = boto3.client('logs', region_name='ap-southeast-2')
lambda_arn = 'arn:aws:lambda:ap-southeast-2:030103857128:function:workbench-matmul'
# Define the log group and log stream names
log_group_name = '/aws/lambda-insights'
last_event_time = [0]
# Define start and end times for the logs (example: last 24 hours)
end_time = int(time.time() * 1000)  # Current time in milliseconds
start_time = end_time - 86400000  # 24 hours ago in milliseconds



response = client.describe_log_streams(
    logGroupName=log_group_name,
    orderBy='LastEventTime',
    descending=True
)

log_streams = response['logStreams']
log_stream_name = []
for log_stream in log_streams:
    # print(log_stream['logStreamName'])
    if log_stream['logStreamName'].startswith(f'{lambda_arn.split(":")[-1]}/') and \
        log_stream['lastIngestionTime'] > last_event_time[0]:
        log_stream_name.append(log_stream['logStreamName'])
        last_event_time.append(log_stream['lastIngestionTime'])
        # print(log_stream['lastIngestionTime'])
        # print(log_stream_name)
        

for lg in log_stream_name:

# Get the log events from the log group and log stream
    response = client.get_log_events(
        logGroupName=log_group_name,
        logStreamName=lg,
        startTime=start_time,
        endTime=end_time
    )

    # Process the log events
    for event in response['events']:
        # Parse the JSON message
        try:
            log_message = json.loads(event['message'])
            cpu_user_time = log_message['cpu_user_time']
            if cpu_user_time == 0:
                print(event['message'])
            else:
                print(cpu_user_time)
        except:
            print("Error parsing log message")

if log_stream_name is not []:
    last_event_time = [max(last_event_time)]
    print(last_event_time)

50
40
20
40
{"cpu_total_time":10,"init_duration":1263,"cold_start":true,"version":"RAM128","total_network":0,"billed_duration":151,"tmp_used":12136448,"function_name":"workbench-matmul","fd_max":1024,"cpu_system_time":10,"tx_bytes":0,"rx_bytes":0,"total_memory":128,"tmp_max":550461440,"tmp_free":538324992,"fd_use":22,"_aws":{"CloudWatchMetrics":[{"Namespace":"LambdaInsights","Dimensions":[["function_name"],["function_name","version"]],"Metrics":[{"Name":"cpu_total_time","Unit":"Milliseconds"},{"Name":"tx_bytes","Unit":"Bytes"},{"Name":"rx_bytes","Unit":"Bytes"},{"Name":"total_network","Unit":"Bytes"},{"Name":"tmp_used","Unit":"Bytes"},{"Name":"memory_utilization","Unit":"Percent"},{"Name":"total_memory","Unit":"Megabytes"},{"Name":"used_memory_max","Unit":"Megabytes"},{"Name":"init_duration","Unit":"Milliseconds"}]}],"Timestamp":1721886601092,"LambdaInsights":{"ShareTelemetry":true}},"cpu_user_time":0,"memory_utilization":85,"duration":150,"agent_memory_avg":8,"billed_mb_ms":19328,"use

In [34]:
import re
log = '''
RequestId: 78ba464e-1d3a-4990-bc07-2c4aa6e5587e Exception: Runtime exited with error: signal: killed
Runtime.ExitError
'''
# Combined regex pattern with named groups
combined_regex = r'Error: (?P<error>.*)|' \
                    r'RequestId:\s+(?P<request_id>[a-f0-9-]+)|' \
                    r'Billed Duration: (?P<duration>\d+) ms'

match = re.search(combined_regex, log)
regex = r'RequestId: (?P<Error>.*)|'\
        r'Exception: (?P<Exception>.*)|'\
        r'error: (?P<error>.*)'
match = re.search(regex, log)
if match.group('Error'):
    print(match.group('Error'))
elif match.group('Exception'):
    print(match.group('Exception'), "exception")
elif match.group('error'):
    print(match.group('error'))

78ba464e-1d3a-4990-bc07-2c4aa6e5587e Exception: Runtime exited with error: signal: killed


In [19]:
import boto3
import json
client =  boto3.client('lambda', region_name='ap-southeast-2')
response = client.invoke(
    FunctionName="arn:aws:lambda:ap-southeast-2:030103857128:function:workbench-matmul",
    InvocationType='RequestResponse',
    LogType='Tail',
    Payload=json.dumps({'n': 10})
)

# Extract Request ID from the response metadata
request_id = response['ResponseMetadata']['RequestId']

In [25]:
for key, value in response['ResponseMetadata'].items():
    print(f'{key}: {value}')

RequestId: 746b7379-90bd-4175-b7c9-7af64b29d13c
HTTPStatusCode: 200
HTTPHeaders: {'date': 'Wed, 24 Jul 2024 05:41:14 GMT', 'content-type': 'application/json', 'content-length': '129', 'connection': 'keep-alive', 'x-amzn-requestid': '746b7379-90bd-4175-b7c9-7af64b29d13c', 'x-amzn-remapped-content-length': '0', 'x-amz-executed-version': '$LATEST', 'x-amz-log-result': 'U1RBUlQgUmVxdWVzdElkOiA3NDZiNzM3OS05MGJkLTQxNzUtYjdjOS03YWY2NGIyOWQxM2MgVmVyc2lvbjogJExBVEVTVAp7ImxldmVsIjoiSU5GTyIsImxvY2F0aW9uIjoiZGVjb3JhdGU6NDUxIiwibWVzc2FnZSI6eyJuIjoxMH0sInRpbWVzdGFtcCI6IjIwMjQtMDctMjQgMDU6NDE6MTQsOTU4KzAwMDAiLCJzZXJ2aWNlIjoibWF0bXVsIiwiY29sZF9zdGFydCI6ZmFsc2UsImZ1bmN0aW9uX25hbWUiOiJ3b3JrYmVuY2gtbWF0bXVsIiwiZnVuY3Rpb25fbWVtb3J5X3NpemUiOiI2NTYiLCJmdW5jdGlvbl9hcm4iOiJhcm46YXdzOmxhbWJkYTphcC1zb3V0aGVhc3QtMjowMzAxMDM4NTcxMjg6ZnVuY3Rpb246d29ya2JlbmNoLW1hdG11bCIsImZ1bmN0aW9uX3JlcXVlc3RfaWQiOiI3NDZiNzM3OS05MGJkLTQxNzUtYjdjOS03YWY2NGIyOWQxM2MiLCJ4cmF5X3RyYWNlX2lkIjoiMS02NmEwOTNmYS0wNjg0Y2VkNzExNmFmYTI0MGY1MjY

In [36]:
log = 'PAYLOAD\tRequestId:\t12-212-1212-21212\tQualifier:\t{alias}\tInput:\t23\n'
regex = r'RequestId:\t([a-f0-9-]+)\t'
match = re.search(regex, log)
if match:
    print(match.group(1))


12-212-1212-21212


In [10]:
import boto3
import json
from datetime import datetime, timedelta, timezone


cloudwatch_logs = boto3.client('logs', region_name='ap-southeast-2')
# Specify the start and end time for the query (example: last 24 hours)
end_time = datetime.now(timezone.utc)
start_time = end_time - timedelta(days=1)
# Define your filter pattern (example: looking for ERROR logs)
# filter_pattern = 'sebs-floatOperation'

# Convert times to milliseconds since the epoch
start_time_ms = int(start_time.timestamp() * 1000)
end_time_ms = int(end_time.timestamp() * 1000)

# Retrieve log events based on the filter pattern and log stream names
while True:
    response = cloudwatch_logs.filter_log_events(
        logGroupName='/aws/lambda-insights',
        startTime=start_time_ms,
        endTime=end_time_ms,
        # filterPattern=filter_pattern
    )
    # Process and print the filtered log events
    for event in response['events']:
        if ((json.loads(event['message'])))['request_id'] == 'edeebc6d-97b8-4ec5-84c6-133191403f58':
            print(event['message'])
    if 'nextToken' in response:
        next_token = response['nextToken']
        response = cloudwatch_logs.filter_log_events(
            logGroupName='/aws/lambda-insights',
            startTime=start_time_ms,
            endTime=end_time_ms,
            # filterPattern=filter_pattern,
            nextToken=next_token
        )
    else:
        break

KeyboardInterrupt: 

In [37]:
import boto3

client = boto3.resource('dynamodb', region_name='ap-southeast-2')

table = client.Table('function_logs')

In [1]:
parsed_events = {
    "0e613505-6b89-4411-aa07-bb801eb1d29a": {
      "cpu_user_time": 20,
      "cpu_system_time": 50,
      "insights_duration": 133,
      "memory_utilisation": 89,
      "billed_duration": 134,
      "billed_mb_ms": 17152,
      "function_name": "workbench-matmul",
      "cold_start": True,
      "insight_init_duration": 1259,
      "used_memory_max": 114,
      "total_memory": 128,
      "total_network": 84,
      "tmp_free": 538324992,
      "tmp_used": 12136448,
      "tx_bytes": 42,
      "fd_max": 1024,
      "rx_bytes": 42,
      "request_id": "0e613505-6b89-4411-aa07-bb801eb1d29a",
      "agent_memory_avg": 8,
      "threads_max": 10,
      "tmp_max": 550461440,
      "agent_memory_max": 9,
      "fd_use": 22,
      "version": "RAM128"
    },
    "77d64206-c667-42f9-a766-fe5b8da8f2e4": {
      "cpu_user_time": 20,
      "cpu_system_time": 50,
      "insights_duration": 20,
      "memory_utilisation": 89,
      "billed_duration": 21,
      "billed_mb_ms": 2688,
      "function_name": "workbench-matmul",
      "cold_start": False,
      "insight_init_duration": 0,
      "used_memory_max": 114,
      "total_memory": 128,
      "total_network": 0,
      "tmp_free": 538320896,
      "tmp_used": 12140544,
      "tx_bytes": 0,
      "fd_max": 1024,
      "rx_bytes": 0,
      "request_id": "77d64206-c667-42f9-a766-fe5b8da8f2e4",
      "agent_memory_avg": 9,
      "threads_max": 10,
      "tmp_max": 550461440,
      "agent_memory_max": 9,
      "fd_use": 21,
      "version": "RAM128"
    }
}

In [49]:
batch = dict(list(parsed_events.items())[:1])
# with table.batch_writer() as batch_writer:
for request_id, log in batch.items():
    # Assuming log is a dictionary with simple key-value pairs
    item = {
        'request_id': request_id,
        **log  # Expanding the log dictionary directly into the item
    }
    print(item)

{'request_id': '0e613505-6b89-4411-aa07-bb801eb1d29a', 'cpu_user_time': 20, 'cpu_system_time': 50, 'insights_duration': 133, 'memory_utilisation': 89, 'billed_duration': 134, 'billed_mb_ms': 17152, 'function_name': 'workbench-matmul', 'cold_start': True, 'insight_init_duration': 1259, 'used_memory_max': 114, 'total_memory': 128, 'total_network': 84, 'tmp_free': 538324992, 'tmp_used': 12136448, 'tx_bytes': 42, 'fd_max': 1024, 'rx_bytes': 42, 'agent_memory_avg': 8, 'threads_max': 10, 'tmp_max': 550461440, 'agent_memory_max': 9, 'fd_use': 22, 'version': 'RAM128'}


In [51]:
len([i for i in range(10, 40, 4)])

8

In [12]:
import boto3

# Assuming parsed_events is defined somewhere
log = parsed_events['0e613505-6b89-4411-aa07-bb801eb1d29a']
del log['request_id']
table = boto3.resource('dynamodb', region_name='ap-southeast-2').Table('function_logs')
request_id = "0e613505-6b89-4411-aa07-bb801eb1d29a"

if isinstance(log, dict):
    # Initialize parts of the update expression
    update_expression_parts = []
    expression_attribute_values = {}
    expression_attribute_names = {}

    # Construct the update expression and attribute values
    for key, value in log.items():
        column_name = key
        placeholder = f':{column_name}'
        update_expression_parts.append(f'#{column_name} = {placeholder}')
        expression_attribute_values[placeholder] = value
        expression_attribute_names[f'#{column_name}'] = key

    # Join the parts to form the complete update expression
    update_expression = 'SET ' + ', '.join(update_expression_parts)
    print(update_expression) 
    print(expression_attribute_values)

    # Correctly pass the constructed ExpressionAttributeNames
    table.update_item(
        Key={'request_id': request_id},
        UpdateExpression=update_expression,
        ExpressionAttributeNames=expression_attribute_names,
        ExpressionAttributeValues=expression_attribute_values
    )

SET #cpu_user_time = :cpu_user_time, #cpu_system_time = :cpu_system_time, #insights_duration = :insights_duration, #memory_utilisation = :memory_utilisation, #billed_duration = :billed_duration, #billed_mb_ms = :billed_mb_ms, #function_name = :function_name, #cold_start = :cold_start, #insight_init_duration = :insight_init_duration, #used_memory_max = :used_memory_max, #total_memory = :total_memory, #total_network = :total_network, #tmp_free = :tmp_free, #tmp_used = :tmp_used, #tx_bytes = :tx_bytes, #fd_max = :fd_max, #rx_bytes = :rx_bytes, #agent_memory_avg = :agent_memory_avg, #threads_max = :threads_max, #tmp_max = :tmp_max, #agent_memory_max = :agent_memory_max, #fd_use = :fd_use, #version = :version
{':cpu_user_time': 20, ':cpu_system_time': 50, ':insights_duration': 133, ':memory_utilisation': 89, ':billed_duration': 134, ':billed_mb_ms': 17152, ':function_name': 'workbench-matmul', ':cold_start': True, ':insight_init_duration': 1259, ':used_memory_max': 114, ':total_memory': 128

In [8]:
log.items()

dict_items([('0e613505-6b89-4411-aa07-bb801eb1d29a', {'cpu_user_time': 20, 'cpu_system_time': 50, 'insights_duration': 133, 'memory_utilisation': 89, 'billed_duration': 134, 'billed_mb_ms': 17152, 'function_name': 'workbench-matmul', 'cold_start': True, 'insight_init_duration': 1259, 'used_memory_max': 114, 'total_memory': 128, 'total_network': 84, 'tmp_free': 538324992, 'tmp_used': 12136448, 'tx_bytes': 42, 'fd_max': 1024, 'rx_bytes': 42, 'request_id': '0e613505-6b89-4411-aa07-bb801eb1d29a', 'agent_memory_avg': 8, 'threads_max': 10, 'tmp_max': 550461440, 'agent_memory_max': 9, 'fd_use': 22, 'version': 'RAM128'}), ('77d64206-c667-42f9-a766-fe5b8da8f2e4', {'cpu_user_time': 20, 'cpu_system_time': 50, 'insights_duration': 20, 'memory_utilisation': 89, 'billed_duration': 21, 'billed_mb_ms': 2688, 'function_name': 'workbench-matmul', 'cold_start': False, 'insight_init_duration': 0, 'used_memory_max': 114, 'total_memory': 128, 'total_network': 0, 'tmp_free': 538320896, 'tmp_used': 12140544, 

In [8]:
start_time = '1721883228514'
log_stream_name = []
while len(log_stream_name) == 0:
    response = client.describe_log_streams(
    logGroupName=log_group_name,
    orderBy='LastEventTime',
    descending=True
    )

    log_streams = response['logStreams']
    
    for log_stream in log_streams:
        if log_stream['logStreamName'].startswith(f'{lambda_arn.split(":")[-1]}/') and \
            end_time >= log_stream['lastIngestionTime'] >= start_time:
            log_stream_name.append(log_stream['logStreamName'])
            last_event_time.append(log_stream['lastIngestionTime'])   
            print(log_stream_name)         
    time.sleep(1)


print(len(log_stream_name))

2024/07/30/workbench-linpack[138]40146703f08d40aa95deb128b41a8765

TypeError: '>=' not supported between instances of 'int' and 'str'

In [9]:
from datetime import datetime, timezone

# Assuming cpu_time is the CPU time in milliseconds
cpu_time = 1722327551720

# Convert milliseconds to seconds
cpu_time_seconds = cpu_time / 1000.0

# Convert to a datetime object in UTC
utc_time = datetime.fromtimestamp(cpu_time_seconds, tz=timezone.utc)

print(utc_time)

2024-07-30 08:19:11.720000+00:00


In [ ]:
txt = ''' 
{'00eefe79-9fe0-4e27-846d-dd7a10933bed': {'duration': 6965, 'init_duration': 1235, 'memory_size': 2304, 'memory_used': 769}, '0126f427-06ec-4192-9ca0-5cce3efb55ec': {'duration': 38748, 'init_duration': None, 'memory_size': 2304, 'memory_used': 2294}, 'bcfa0b8c-8862-40c5-915b-a041930792ef': {'duration': 23580, 'init_duration': None, 'memory_size': 2304, 'memory_used': 2294}, '5ce488ce-ed88-4723-9ed0-1e24080425bd': {'duration': 20308, 'init_duration': None, 'memory_size': 2304, 'memory_used': 2294}, '3e695031-f571-4322-ba4a-669637f697cb': {'duration': 18150, 'init_duration': None, 'memory_size': 2304, 'memory_used': 2294}, '42948c65-fe54-4485-a5fa-af8af019ef58': {'duration': 20605, 'init_duration': None, 'memory_size': 2304, 'memory_used': 2294}, '75d77342-afa0-413a-8328-2c7eda8c3f51': {'duration': 39082, 'init_duration': None, 'memory_size': 2304, 'memory_used': 2304}, 'c5e779cc-bd15-4410-a06b-a4a5d9e8ab45': {'duration': 63, 'init_duration': 1182, 'memory_size': 3008, 'memory_used': 135}, 'ff61697e-abf8-4602-8e83-91986d4b0323': {'duration': 64, 'init_duration': None, 'memory_size': 3008, 'memory_used': 140}, 'a414bc88-dd87-4113-95b5-646a48a66197': {'duration': 751, 'init_duration': None, 'memory_size': 3008, 'memory_used': 276}, 'cfce028f-45c3-4a3e-965f-d639a91791a5': {'duration': 295, 'init_duration': None, 'memory_size': 3008, 'memory_used': 276}, '69cda233-5aed-42eb-ab9f-e8faf45abd82': {'duration': 1898, 'init_duration': None, 'memory_size': 3008, 'memory_used': 438}, '46b0ab88-1001-46e0-a53b-d571b4da9084': {'duration': 855, 'init_duration': None, 'memory_size': 3008, 'memory_used': 438}, '87f8ff7b-896e-4e56-a22b-3ffb1c08e4b1': {'duration': 264, 'init_duration': None, 'memory_size': 3008, 'memory_used': 438}, '2a4bb9f5-8e25-4e5c-8040-205d3bb74570': {'duration': 2875, 'init_duration': None, 'memory_size': 3008, 'memory_used': 550}, '3e547e48-fec6-415c-b90a-a706c135792e': {'duration': 4152, 'init_duration': None, 'memory_size': 3008, 'memory_used': 678}, '8dc5d88b-2d1c-4368-b330-2062707f700f': {'duration': 215, 'init_duration': None, 'memory_size': 3008, 'memory_used': 678}, 'e6e8f728-73bb-43db-beb2-bdd3da5e077d': {'duration': 348, 'init_duration': None, 'memory_size': 3008, 'memory_used': 678}, '0fc46369-4932-4d01-a42d-9fdfc4ffed2f': {'duration': 399, 'init_duration': None, 'memory_size': 3008, 'memory_used': 678}, '071789b9-ff2a-4c38-b68d-e10cbbc5476b': {'duration': 435, 'init_duration': None, 'memory_size': 3008, 'memory_used': 678}, '43a13aed-c604-4641-ae52-498b237d840d': {'duration': 2160, 'init_duration': None, 'memory_size': 3008, 'memory_used': 678}, 'c65591bc-45f7-4683-8515-ef4a1520b8aa': {'duration': 2513, 'init_duration': None, 'memory_size': 3008, 'memory_used': 678}, '0747454e-2660-4998-b595-6a6bc5709b50': {'duration': 3193, 'init_duration': None, 'memory_size': 3008, 'memory_used': 678}, '882cdf46-7a17-49a3-abac-661e7eb8773b': {'duration': 4341, 'init_duration': None, 'memory_size': 3008, 'memory_used': 702}, '3ef337db-bfa0-4ea2-9079-8e290258a5ed': {'duration': 1476, 'init_duration': None, 'memory_size': 3008, 'memory_used': 702}, '3c98e787-7dcd-4e25-b82b-3f4dc46ad684': {'duration': 3830, 'init_duration': None, 'memory_size': 3008, 'memory_used': 702}, '2b0307b5-0139-48db-baea-cfa76eb8747d': {'duration': 1844, 'init_duration': None, 'memory_size': 3008, 'memory_used': 702}, 'b21a58a4-43a5-4c44-a457-a36efa2d6501': {'duration': 2192, 'init_duration': None, 'memory_size': 3008, 'memory_used': 702}, 'cddca4da-f1a1-4aeb-a637-772ad248cc7e': {'duration': 2612, 'init_duration': None, 'memory_size': 3008, 'memory_used': 702}, '1ec83ef5-3cd6-442c-b909-5b3a1cd6d15f': {'duration': 3016, 'init_duration': None, 'memory_size': 3008, 'memory_used': 702}, '8f72629b-e865-4528-8258-2b500d086d19': {'duration': 3598, 'init_duration': None, 'memory_size': 3008, 'memory_used': 702}, 'd607c2e6-104e-482b-baf4-3287d055457b': {'duration': 3866, 'init_duration': None, 'memory_size': 3008, 'memory_used': 702}, 'd6cf4033-696d-4132-b43d-de05cc4f7d50': {'duration': 12732, 'init_duration': None, 'memory_size': 3008, 'memory_used': 1322}, 'a4c8c9db-3094-40a7-b931-b6037afdf8c1': {'duration': 24676, 'init_duration': None, 'memory_size': 3008, 'memory_used': 2037}, 'ede1b59b-aff9-4073-9a90-6766f36bae5b': {'duration': 15608, 'init_duration': None, 'memory_size': 3008, 'memory_used': 2037}, '97a357ca-cdfa-4004-a6a6-d46ce487dcab': {'duration': 22410, 'init_duration': None, 'memory_size': 3008, 'memory_used': 2037}, 'd687d384-41c1-4bf1-84f0-0e54f2cafcee': {'duration': 27420, 'init_duration': None, 'memory_size': 3008, 'memory_used': 2209}, '54acf05b-f78e-414e-9bf1-c0e849ad5a86': {'duration': 63, 'init_duration': 1205, 'memory_size': 2944, 'memory_used': 131}, '4590066a-4982-4d9e-a075-8359e25cccef': {'duration': 464, 'init_duration': None, 'memory_size': 2944, 'memory_used': 217}, '35f83744-3875-4a81-8ef6-3e7e74e1cbff': {'duration': 31, 'init_duration': None, 'memory_size': 2944, 'memory_used': 217}, '686477ec-8e81-4a04-a975-6976c874ce26': {'duration': 30, 'init_duration': None, 'memory_size': 2944, 'memory_used': 217}, '8d4cd162-8664-4170-9c15-509ae4d53898': {'duration': 582, 'init_duration': None, 'memory_size': 2944, 'memory_used': 250}, 'd8f50e66-8ab2-49c2-b3c6-fd5dde0965a0': {'duration': 74, 'init_duration': None, 'memory_size': 2944, 'memory_used': 250}, '8b1f503d-a7b8-419f-b91e-cb93da2bba75': {'duration': 657, 'init_duration': None, 'memory_size': 2944, 'memory_used': 263}, '36fd0def-a63d-40f1-8eae-1484324ac6c6': {'duration': 716, 'init_duration': None, 'memory_size': 2944, 'memory_used': 275}, '011ffa77-5168-4707-bed7-c891e6ea8de9': {'duration': 238, 'init_duration': None, 'memory_size': 2944, 'memory_used': 275}, 'f03908ae-5fd9-4aee-a1c3-ddace066da98': {'duration': 1056, 'init_duration': None, 'memory_size': 2944, 'memory_used': 327}, '982b174e-0724-4929-a9dc-103e8c51a9b6': {'duration': 1178, 'init_duration': None, 'memory_size': 2944, 'memory_used': 341}, '06cae2ae-740d-4cf8-af6d-3464e8b72d49': {'duration': 450, 'init_duration': None, 'memory_size': 2944, 'memory_used': 341}, 'b8c1e506-f44f-4165-b74d-b4ff82df382b': {'duration': 1402, 'init_duration': None, 'memory_size': 2944, 'memory_used': 371}, '1855ee89-2ad2-42fe-9023-612c66839855': {'duration': 2, 'init_duration': None, 'memory_size': 2944, 'memory_used': 371}, '51739b6a-87a1-47c2-8c38-32649c55c409': {'duration': 572, 'init_duration': None, 'memory_size': 2944, 'memory_used': 371}, 'b6053df2-1cbb-4f01-b7c0-10a210f00eb7': {'duration': 1641, 'init_duration': None, 'memory_size': 2944, 'memory_used': 403}, '45628074-082b-4e26-bae0-abb27d6f60f6': {'duration': 1927, 'init_duration': None, 'memory_size': 2944, 'memory_used': 437}, '3890f047-452a-4f8c-9dde-c1042d405eb4': {'duration': 881, 'init_duration': None, 'memory_size': 2944, 'memory_used': 437}, '6d7a9d68-153f-4135-a5d3-33a4765280ab': {'duration': 261, 'init_duration': None, 'memory_size': 2944, 'memory_used': 437}, '25ddc843-6c0c-41c0-8e4f-d948d7fb0ad6': {'duration': 1039, 'init_duration': None, 'memory_size': 2944, 'memory_used': 437}, '5119d62f-4e5f-46a0-b562-e199044422e6': {'duration': 3159, 'init_duration': None, 'memory_size': 2944, 'memory_used': 567}, '75c9f8ab-695c-4003-82fb-e7df2538051e': {'duration': 4312, 'init_duration': None, 'memory_size': 2944, 'memory_used': 675}, 'b8988739-c765-4a37-aa95-225f136f40a1': {'duration': 352, 'init_duration': None, 'memory_size': 2944, 'memory_used': 675}, 'f1f6d0a5-d523-4bae-a925-37489554c2b0': {'duration': 399, 'init_duration': None, 'memory_size': 2944, 'memory_used': 675}, '53400b48-d6a5-46b7-8f6a-e7acebec88bb': {'duration': 2366, 'init_duration': None, 'memory_size': 2944, 'memory_used': 675}, 'b6ef4900-048e-440f-91ba-7793d9adef6a': {'duration': 2580, 'init_duration': None, 'memory_size': 2944, 'memory_used': 675}, 'bbb5706a-78d2-4020-b750-6ab82f020d97': {'duration': 797, 'init_duration': None, 'memory_size': 2944, 'memory_used': 675}, '6814ecd2-63c8-4b40-ad66-86860f875249': {'duration': 1064, 'init_duration': None, 'memory_size': 2944, 'memory_used': 675}, 'd662426a-7571-490e-af9d-fb3615b55597': {'duration': 3177, 'init_duration': None, 'memory_size': 2944, 'memory_used': 675}, '3d383083-7c0c-42fc-b082-bd4f027bb5aa': {'duration': 1415, 'init_duration': None, 'memory_size': 2944, 'memory_used': 675}, 'f832b935-4c93-430a-b450-37c65718dae2': {'duration': 4780, 'init_duration': None, 'memory_size': 2944, 'memory_used': 723}, '091d81d7-81fa-46d0-a610-de5181c497cf': {'duration': 2347, 'init_duration': None, 'memory_size': 2944, 'memory_used': 723}, '66beb3f3-b6b2-41b3-abb7-424711df25b4': {'duration': 1895, 'init_duration': None, 'memory_size': 2944, 'memory_used': 723}, 'e451d7e2-12d9-4335-8a7b-69c022ded6f0': {'duration': 5649, 'init_duration': None, 'memory_size': 2944, 'memory_used': 795}, '4ab44b77-790c-43b6-b1f4-d7a03b39498d': {'duration': 9214, 'init_duration': None, 'memory_size': 2944, 'memory_used': 1067}, '056837b5-8366-408c-96a2-e4b1c95c5e03': {'duration': 3754, 'init_duration': None, 'memory_size': 2944, 'memory_used': 1067}, '28b06413-49c3-4c2a-a070-85dfb4bbdf75': {'duration': 4019, 'init_duration': None, 'memory_size': 2944, 'memory_used': 1067}, 'b1dd6c57-abb3-4bb4-869a-1ea36050c508': {'duration': 13746, 'init_duration': None, 'memory_size': 2944, 'memory_used': 1352}, '57cc5dc2-55f2-443c-b6ea-5287897a35c4': {'duration': 24695, 'init_duration': None, 'memory_size': 2944, 'memory_used': 1992}, '2e42438f-864e-46bd-8b00-b4f6a1c4422d': {'duration': 22170, 'init_duration': None, 'memory_size': 2944, 'memory_used': 1992}, '63f24e38-c199-427a-a81a-0f86dae5736a': {'duration': 27840, 'init_duration': None, 'memory_size': 2944, 'memory_used': 2162}, '5873f455-6482-4229-bd51-d436c7135d67': {'duration': 21305, 'init_duration': None, 'memory_size': 2944, 'memory_used': 2162}, '36396b38-c3db-4b83-b333-82cc79ee4287': {'duration': 17340, 'init_duration': None, 'memory_size': 2944, 'memory_used': 2162}, '17f94640-d595-4a74-b459-0d1534a1cfe0': {'duration': 28672, 'init_duration': None, 'memory_size': 2944, 'memory_used': 2207}, '9b039e96-1b4a-49a7-8457-ce275ea82a26': {'duration': 277, 'init_duration': 1230, 'memory_size': 2944, 'memory_used': 182}, '1be9e388-6876-47ea-8681-03cfc2353ff0': {'duration': 1032, 'init_duration': None, 'memory_size': 2944, 'memory_used': 313}, 'badc7ac2-b03b-41f2-81a4-4effdc2efcba': {'duration': 4, 'init_duration': None, 'memory_size': 2944, 'memory_used': 313}, 'ed5298e3-4c39-460c-85d4-a4c526007931': {'duration': 3, 'init_duration': None, 'memory_size': 2944, 'memory_used': 313}, '902ab417-3a9f-42fd-be2d-b0965e9508e1': {'duration': 7, 'init_duration': None, 'memory_size': 2944, 'memory_used': 313}, '05cef79a-be07-4ffe-9fac-e7f7a9ffe70e': {'duration': 9, 'init_duration': None, 'memory_size': 2944, 'memory_used': 313}, '7f73ed37-0ec0-416f-91f8-f1fb14bfaba2': {'duration': 3703, 'init_duration': None, 'memory_size': 2944, 'memory_used': 613}, '85ca8401-2519-40ae-9891-00d5aa3fbd24': {'duration': 60, 'init_duration': None, 'memory_size': 2944, 'memory_used': 613}, 'c1ec8198-a1d6-4d9d-964d-00140acd0452': {'duration': 67, 'init_duration': None, 'memory_size': 2944, 'memory_used': 613}, 'f7f7553b-3b27-4202-944f-8a71e50baa59': {'duration': 128, 'init_duration': None, 'memory_size': 2944, 'memory_used': 613}, 'c803b91a-16d0-42e9-a9f0-dc4aa2811747': {'duration': 303, 'init_duration': None, 'memory_size': 2944, 'memory_used': 613}, 'c7da1c6c-6b60-49ff-8003-091e4e2a25f6': {'duration': 225, 'init_duration': None, 'memory_size': 2944, 'memory_used': 613}, '3e1d0f1c-f4fa-4060-81e0-e02e66973d0d': {'duration': 2036, 'init_duration': None, 'memory_size': 2944, 'memory_used': 613}, '450d8c08-1867-40f8-a84a-e4f137c799ee': {'duration': 452, 'init_duration': None, 'memory_size': 2944, 'memory_used': 613}, 'e422362b-725b-4692-a6de-f0708cfc2941': {'duration': 2188, 'init_duration': None, 'memory_size': 2944, 'memory_used': 613}, 'a90d2911-e1f6-4e1a-8a13-26aef1d89f8a': {'duration': 517, 'init_duration': None, 'memory_size': 2944, 'memory_used': 613}, '734a1d40-dee2-4e67-8a3f-8ce23e659376': {'duration': 639, 'init_duration': None, 'memory_size': 2944, 'memory_used': 613}, 'b68d822e-f810-426a-a06b-344038b2a0c0': {'duration': 703, 'init_duration': None, 'memory_size': 2944, 'memory_used': 613}, '5ee88a61-5b53-440f-922d-c1acdb79b8d9': {'duration': 3527, 'init_duration': None, 'memory_size': 2944, 'memory_used': 613}, 'bb4a2a3f-c106-4d3b-8896-39502c4a204e': {'duration': 3864, 'init_duration': None, 'memory_size': 2944, 'memory_used': 633}, '73083d71-5cfa-4c8d-bce4-a9475afe82b1': {'duration': 1631, 'init_duration': None, 'memory_size': 2944, 'memory_used': 633}, 'f9c5de50-8311-4e46-b6fb-b4d81e3a4328': {'duration': 7362, 'init_duration': None, 'memory_size': 2944, 'memory_used': 926}, 'efe29467-3611-4298-bfa4-991a107975c9': {'duration': 8389, 'init_duration': None, 'memory_size': 2944, 'memory_used': 1011}, '8f55db50-701c-42ae-952c-f99735767015': {'duration': 20523, 'init_duration': None, 'memory_size': 2944, 'memory_used': 1752}, '851e9780-ab68-4e80-b246-75f920abb040': {'duration': 29113, 'init_duration': None, 'memory_size': 2944, 'memory_used': 2206}, '35e56a3a-e431-43ec-9d96-75b2ec67f383': {'duration': 15573, 'init_duration': None, 'memory_size': 2944, 'memory_used': 2206}, '850d4665-ba28-48eb-889c-a462cb529d4e': {'duration': 15031, 'init_duration': None, 'memory_size': 2944, 'memory_used': 2206}, '9eee002e-df51-40a6-9512-101a8325df59': {'duration': 26970, 'init_duration': None, 'memory_size': 2944, 'memory_used': 2206}, 'ad3d76d2-5779-47d3-8745-fd7121086986': {'duration': 13860, 'init_duration': None, 'memory_size': 2944, 'memory_used': 2206}, '488775ea-3bea-4286-a26c-25b74fd83c33': {'duration': 22799, 'init_duration': None, 'memory_size': 2944, 'memory_used': 2206}, '180605ec-6cde-4cee-89e2-2003a89d4858': {'duration': 31153, 'init_duration': None, 'memory_size': 2944, 'memory_used': 2341}, 'f049dc5a-4bea-43f6-828c-04f8582fb654': {'duration': 38, 'init_duration': 1202, 'memory_size': 2944, 'memory_used': 127}, 'a925fbab-937c-4f98-9824-7af6333e5f6e': {'duration': 29903, 'init_duration': None, 'memory_size': 2944, 'memory_used': 2252}, '8f119e2e-d5f6-48dc-afba-666336349203': {'duration': 5122, 'init_duration': None, 'memory_size': 2944, 'memory_used': 2252}, 'bfc156bd-8e90-4d24-aca8-33248cd9162d': {'duration': 18842, 'init_duration': None, 'memory_size': 2944, 'memory_used': 2252}, '6ab4fafb-b41e-407a-b16d-b9e9ff26515e': {'duration': 10594, 'init_duration': None, 'memory_size': 2944, 'memory_used': 2252}, 'eff835ef-b28e-498d-8209-c6d3855ceaf1': {'duration': 11406, 'init_duration': None, 'memory_size': 2944, 'memory_used': 2252}, '3ddc093f-9c86-4546-b3ce-5dececd91084': {'duration': 13046, 'init_duration': None, 'memory_size': 2944, 'memory_used': 2252}, '2b9b1a29-e1f7-4a65-92ec-1d2ea0203f4d': {'duration': 31331, 'init_duration': None, 'memory_size': 2944, 'memory_used': 2388}, '8def5b25-3397-4e10-8ef4-ae42e0cf46b8': {'duration': 27365, 'init_duration': None, 'memory_size': 2944, 'memory_used': 2388}, '56538392-5ef8-4777-a601-99e69cf03641': {'duration': 10, 'init_duration': 1029, 'memory_size': 2944, 'memory_used': 110}, 'b8e9f06b-3cc9-4230-8bd6-56ea14b6c02d': {'duration': 191, 'init_duration': None, 'memory_size': 2944, 'memory_used': 167}, 'f99a17d0-fd45-432e-851a-ac0870322c86': {'duration': 18639, 'init_duration': None, 'memory_size': 2944, 'memory_used': 1713}, 'a3116613-aa99-4d08-aa6b-8d6beb56acde': {'duration': 7667, 'init_duration': None, 'memory_size': 2944, 'memory_used': 1713}, 'bf7cc09a-a6a5-4ce6-a4f3-c25ae6a73d73': {'duration': 8441, 'init_duration': None, 'memory_size': 2944, 'memory_used': 1713}, 'e8ba94bb-c41f-4f86-87fb-34c696ba9a51': {'duration': 8055, 'init_duration': None, 'memory_size': 2944, 'memory_used': 1713}, '91244f43-eb12-4e13-af72-cd3c6b61bca8': {'duration': 8724, 'init_duration': None, 'memory_size': 2944, 'memory_used': 1713}, '62221d18-c5a4-49d9-852d-b479f1b788da': {'duration': 9337, 'init_duration': None, 'memory_size': 2944, 'memory_used': 1713}, 'eead6b2f-fe4d-4a14-820a-bee6dfc6cd24': {'duration': 26324, 'init_duration': None, 'memory_size': 2944, 'memory_used': 2206}, '5c1428e5-9b95-4b1f-b967-9c3baeb42413': {'duration': 13345, 'init_duration': None, 'memory_size': 2944, 'memory_used': 2206}, '4b036423-34b5-49dd-b890-19ddb9c46ef9': {'duration': 19036, 'init_duration': None, 'memory_size': 2944, 'memory_used': 2206}, 'cbcabe1e-fb7d-4c39-8827-924eca953b00': {'duration': 24047, 'init_duration': None, 'memory_size': 2944, 'memory_used': 2206}, 'df4cc3a0-9ac4-4b20-b2ec-40c16bddac80': {'duration': 72, 'init_duration': 1243, 'memory_size': 2304, 'memory_used': 132}, 'a6aff1ee-7047-4ef4-b8ce-f4ba46f70134': {'duration': 636, 'init_duration': None, 'memory_size': 2304, 'memory_used': 229}, '7c1c78aa-8ea6-4460-b184-e339ec33a007': {'duration': 741, 'init_duration': None, 'memory_size': 2304, 'memory_used': 251}, '1217cfbe-72ed-4cef-a9b4-d1250a56e7ce': {'duration': 92, 'init_duration': None, 'memory_size': 2304, 'memory_used': 251}, 'e9012e0a-71fb-4e3b-9076-e2f6daecc0b8': {'duration': 887, 'init_duration': None, 'memory_size': 2304, 'memory_used': 276}, '3648aeaa-5b7e-4511-aa3c-35fb4dec8644': {'duration': 218, 'init_duration': None, 'memory_size': 2304, 'memory_used': 276}, 'ada564e3-11d8-4132-9092-0996d51e1fd0': {'duration': 145, 'init_duration': None, 'memory_size': 2304, 'memory_used': 276}, '3a598c74-86a3-43b5-9038-649da07098ee': {'duration': 304, 'init_duration': None, 'memory_size': 2304, 'memory_used': 276}, '2c178ced-ba82-47ce-8ea6-be8af07cd403': {'duration': 320, 'init_duration': None, 'memory_size': 2304, 'memory_used': 276}, '5a8efc51-eb85-4a1d-aa06-5d44e2952df6': {'duration': 425, 'init_duration': None, 'memory_size': 2304, 'memory_used': 276}, '530cee5a-a80f-4817-8970-94a92edcf2fb': {'duration': 1941, 'init_duration': None, 'memory_size': 2304, 'memory_used': 386}, '0f8168bb-de03-423a-a678-a60013107708': {'duration': 4, 'init_duration': None, 'memory_size': 2304, 'memory_used': 386}, 'bb2708be-934c-4b1f-9b52-c1fd61497375': {'duration': 2429, 'init_duration': None, 'memory_size': 2304, 'memory_used': 437}, '2dde7b99-bf40-4d77-85f4-4d3c21e00a5d': {'duration': 3475, 'init_duration': None, 'memory_size': 2304, 'memory_used': 528}, 'ca83baa3-4a6d-4d7a-936c-0dae1b6cb321': {'duration': 1331, 'init_duration': None, 'memory_size': 2304, 'memory_used': 528}, 'f7eb9df9-3dd4-4f16-ae2b-571fcb855687': {'duration': 1464, 'init_duration': None, 'memory_size': 2304, 'memory_used': 528}, 'a99f873b-e5f4-4fad-a868-496cf82ba342': {'duration': 5, 'init_duration': None, 'memory_size': 2304, 'memory_used': 528}, 'a8307e7d-b879-4a8f-8e56-9e8776db88c2': {'duration': 2, 'init_duration': None, 'memory_size': 2304, 'memory_used': 528}, 'edf1f489-1d54-4f07-bc40-06d8b6847880': {'duration': 7, 'init_duration': None, 'memory_size': 2304, 'memory_used': 528}, 'e67d1368-9558-4e29-bfb4-8c7a61298427': {'duration': 8, 'init_duration': None, 'memory_size': 2304, 'memory_used': 528}, 'c8a7cb06-c503-4ce3-bd88-35c4865bf2e9': {'duration': 19, 'init_duration': None, 'memory_size': 2304, 'memory_used': 528}, '0b6592d3-0457-454c-ba76-3f81e6674e43': {'duration': 499, 'init_duration': None, 'memory_size': 2304, 'memory_used': 528}, 'fa3d9e4b-c07b-4f1c-85e8-a150356585a7': {'duration': 6084, 'init_duration': None, 'memory_size': 2304, 'memory_used': 730}, 'd5c6ebe7-0157-46d4-8c42-29e02cb6055a': {'duration': 2028, 'init_duration': None, 'memory_size': 2304, 'memory_used': 730}, '0b22001e-893d-4305-bffb-706a27b70d26': {'duration': 10194, 'init_duration': None, 'memory_size': 2304, 'memory_used': 990}, '8dcbc586-e23e-427c-b5fd-8c33bbe210d0': {'duration': 25790, 'init_duration': None, 'memory_size': 2304, 'memory_used': 1760}, 'e682d6da-b175-4d30-8723-ecafc2b4bc04': {'duration': 8277, 'init_duration': None, 'memory_size': 2304, 'memory_used': 1760}, '3c7e70c9-dfce-43ed-8512-7c5e3ea9c0ce': {'duration': 39939, 'init_duration': None, 'memory_size': 2304, 'memory_used': 2304}, '9b3d94a1-30e0-4f9f-91c2-e99cc8b81456': {'duration': 30140, 'init_duration': None, 'memory_size': 2304, 'memory_used': 2304}, '09412d68-5cc0-49b9-9bb5-2b35ae9d519f': {'duration': 28946, 'init_duration': None, 'memory_size': 2304, 'memory_used': 2304}, '1d5d113c-01da-4236-a59a-a0b6e9fd9946': {'duration': 38377, 'init_duration': None, 'memory_size': 2304, 'memory_used': 2304}, '985e4508-fcad-4976-805e-fe30c1e2957c': {'duration': 30, 'init_duration': 1210, 'memory_size': 2688, 'memory_used': 125}, '1dd64106-2217-46bc-bdee-9edc41ffe03d': {'duration': 5330, 'init_duration': None, 'memory_size': 2688, 'memory_used': 725}, '8b21d8eb-4cbc-40ac-b22b-ec2f4e599ef5': {'duration': 17570, 'init_duration': None, 'memory_size': 2688, 'memory_used': 1531}, '4a63c1ae-07b3-4a46-b730-5742a03ce155': {'duration': 33405, 'init_duration': None, 'memory_size': 2688, 'memory_used': 2298}, '2c36957d-e05b-4dac-b127-9c41bd28c07e': {'duration': 14932, 'init_duration': None, 'memory_size': 2688, 'memory_used': 2298}, '477fe932-0907-421a-aa8f-13b6d9c55475': {'duration': 23254, 'init_duration': None, 'memory_size': 2688, 'memory_used': 2298}, '39780338-f552-4281-95e9-b330ace9665e': {'duration': 33112, 'init_duration': None, 'memory_size': 2688, 'memory_used': 2300}, '7b52ee0d-a76f-48f7-8ab8-f2b2f2198ca9': {'duration': 21103, 'init_duration': 1229, 'memory_size': 2432, 'memory_used': 1562}, '88ee9002-c38d-4fcd-8443-1db97cc236e8': {'duration': 12582, 'init_duration': None, 'memory_size': 2432, 'memory_used': 1562}, 'ced2dc3b-ecc6-441c-9b95-63be934cf872': {'duration': 6338, 'init_duration': None, 'memory_size': 2432, 'memory_used': 1562}, 'cb3eb821-ff38-4177-9a4f-d4a0a87a4ad4': {'duration': 25919, 'init_duration': None, 'memory_size': 2432, 'memory_used': 1830}, 'f7249616-ecfd-4c6d-9e41-d0a6268146b0': {'duration': 19349, 'init_duration': None, 'memory_size': 2432, 'memory_used': 1830}, '55636a7c-e7ee-493c-9c41-80e75e67f646': {'duration': 18158, 'init_duration': None, 'memory_size': 2432, 'memory_used': 1830}, '9fbb1210-de46-4091-8a56-e22e1f9ba05e': {'duration': 36067, 'init_duration': None, 'memory_size': 2432, 'memory_used': 2250}, '691d7097-a3d0-4e77-ac68-4788210e97d6': {'duration': 16, 'init_duration': 1241, 'memory_size': 2560, 'memory_used': 118}, '41e0f439-1121-4a0b-951e-89e04cf58d55': {'duration': 20092, 'init_duration': None, 'memory_size': 2560, 'memory_used': 1566}, 'c950ec70-b571-464e-a9ff-df70ccef1c31': {'duration': 37591, 'init_duration': None, 'memory_size': 2560, 'memory_used': 2388}, '86ff2fc5-bc8e-4aa0-ac0a-a8fc4c77f683': {'duration': 8337, 'init_duration': None, 'memory_size': 2560, 'memory_used': 2388}, 'fe78a793-b188-40cf-b3d6-1c7c3d36ecba': {'duration': 8669, 'init_duration': None, 'memory_size': 2560, 'memory_used': 2388}, 'a02c9e1d-a034-4452-8a3f-af8a5e7bd5bf': {'duration': 33276, 'init_duration': None, 'memory_size': 2560, 'memory_used': 2388}, 'ff763d1f-8e9e-434d-b75b-96d35d774836': {'duration': 14841, 'init_duration': None, 'memory_size': 2560, 'memory_used': 2388}, 'f3cd139d-04ee-4682-8d76-84ab624b744b': {'duration': 17272, 'init_duration': None, 'memory_size': 2560, 'memory_used': 2388}, '3025ae67-460b-4032-9648-528a8ef11f9d': {'duration': 34051, 'init_duration': None, 'memory_size': 2560, 'memory_used': 2388}, '40b1cb2f-4ecf-4dcd-bfb6-36625b544dd8': {'duration': 5276, 'init_duration': 1255, 'memory_size': 2176, 'memory_used': 634}, '274a6027-9494-4737-bb42-05decf5411f8': {'duration': 19860, 'init_duration': None, 'memory_size': 2176, 'memory_used': 1425}, 'd6e8f982-2a3c-4cff-8aaf-3c16e9b5b332': {'duration': 30224, 'init_duration': None, 'memory_size': 2176, 'memory_used': 1874}, None: {'function_error': 'Runtime exited with error: signal: killed'}, 'd6129cef-4cc8-42c6-b327-c9c068f4a841': {'duration': 4809, 'init_duration': None, 'memory_size': 2176, 'memory_used': 2176}, 'b26c35ac-7235-4707-a19c-13b1832a635e': {'duration': 31072, 'init_duration': None, 'memory_size': 2176, 'memory_used': 1864}, '7d3b8789-1951-4481-a317-80c16ea0dff5': {'duration': 38288, 'init_duration': None, 'memory_size': 2176, 'memory_used': 2176}, 'f96d84de-9f6c-40d6-acc2-01b5ef82ff9d': {'duration': 25581, 'init_duration': 1259, 'memory_size': 2816, 'memory_used': 1909}, '323a171c-d214-4b57-bc8e-4e665e01e8dc': {'duration': 5634, 'init_duration': None, 'memory_size': 2816, 'memory_used': 1909}, 'bffbefea-9610-4bfc-a7b0-8f3be37c0883': {'duration': 13756, 'init_duration': None, 'memory_size': 2816, 'memory_used': 1909}, '2c5e21b7-723d-42be-93fb-9798e0c5c1fd': {'duration': 18700, 'init_duration': None, 'memory_size': 2816, 'memory_used': 1909}, 'df005f6b-5f9d-447f-853b-81d483ddf37b': {'duration': 28150, 'init_duration': None, 'memory_size': 2816, 'memory_used': 2078}, '3fff433c-22a5-427a-b12f-00a40931219e': {'duration': 16061, 'init_duration': None, 'memory_size': 2816, 'memory_used': 2078}, '0019f754-5a3b-4e73-adbf-900add592929': {'duration': 31839, 'init_duration': None, 'memory_size': 2816, 'memory_used': 2251}, 'd4f881db-fdf6-4dee-a517-6f8540151f80': {'duration': 22947, 'init_duration': 1210, 'memory_size': 2944, 'memory_used': 1873}, '036acccf-b71f-44eb-9cd3-39d2df67e29b': {'duration': 4790, 'init_duration': None, 'memory_size': 2944, 'memory_used': 1873}, '655ab07c-64c1-4b7a-a13d-fd2740c0e86e': {'duration': 31973, 'init_duration': None, 'memory_size': 2944, 'memory_used': 2388}, 'bbbd6c90-827e-4ade-b0dd-828d31231c18': {'duration': 17729, 'init_duration': None, 'memory_size': 2944, 'memory_used': 2388}, '30b73a82-4d8e-4d23-a5be-d8caa993c5cd': {'duration': 18329, 'init_duration': None, 'memory_size': 2944, 'memory_used': 2388}, '8112e02c-9e76-4eb9-babf-7b04ca8fa449': {'duration': 24537, 'init_duration': None, 'memory_size': 2944, 'memory_used': 2388}, '58aa386f-5267-4394-93bd-74af5a71ef84': {'duration': 16, 'init_duration': 1204, 'memory_size': 2688, 'memory_used': 117}, 'fdb78747-0ee6-45f3-9032-b4233d300b73': {'duration': 442, 'init_duration': None, 'memory_size': 2688, 'memory_used': 211}, '346e9cf3-6888-41e5-b5cb-0d0a6e34ba1f': {'duration': 12333, 'init_duration': None, 'memory_size': 2688, 'memory_used': 1192}, 'abc1a6d8-9f6f-490d-94a4-925c9b03cf4c': {'duration': 30072, 'init_duration': None, 'memory_size': 2688, 'memory_used': 2122}, '591a48e6-1906-48e7-bb87-bb1a88b4031b': {'duration': 9460, 'init_duration': None, 'memory_size': 2688, 'memory_used': 2122}, '74927b2e-a19c-47e2-a17b-f6d33e80f947': {'duration': 8224, 'init_duration': None, 'memory_size': 2688, 'memory_used': 2122}, 'fe19ef22-96e0-4678-9f31-08a47c9342f6': {'duration': 31281, 'init_duration': None, 'memory_size': 2688, 'memory_used': 2210}, '0f8e15f9-d29c-4187-b743-2fe828a08f4a': {'duration': 14449, 'init_duration': None, 'memory_size': 2688, 'memory_used': 2210}, '47f7be9c-08ff-4a1e-9b8e-d1e831e49338': {'duration': 21906, 'init_duration': None, 'memory_size': 2688, 'memory_used': 2210}, '79c8db66-a251-42db-8f98-962e45f314b3': {'duration': 25993, 'init_duration': None, 'memory_size': 2688, 'memory_used': 2210}, '2c089822-5297-418c-823d-20a40a3ca782': {'duration': 151, 'init_duration': 1234, 'memory_size': 2560, 'memory_used': 156}, '88fb644d-c177-4e9b-9cd2-43bb00f5bc21': {'duration': 299, 'init_duration': None, 'memory_size': 2560, 'memory_used': 187}, 'dbbd8d0c-10d4-4427-b6bb-404b051d895d': {'duration': 302, 'init_duration': None, 'memory_size': 2560, 'memory_used': 194}, '1e155270-5903-47de-b934-6d868a7c9ea1': {'duration': 419, 'init_duration': None, 'memory_size': 2560, 'memory_used': 213}, '1997ef3e-7be1-4411-9a6f-28df87357bc7': {'duration': 511, 'init_duration': None, 'memory_size': 2560, 'memory_used': 234}, '0d05f0dd-53d6-4faa-ad19-796ad85e4d5f': {'duration': 1797, 'init_duration': None, 'memory_size': 2560, 'memory_used': 390}, 'da8857cc-27a6-4078-a5af-add9bbf3f128': {'duration': 4, 'init_duration': None, 'memory_size': 2560, 'memory_used': 390}, '962b0746-ee33-4f2d-a4c0-a6d869d88377': {'duration': 1877, 'init_duration': None, 'memory_size': 2560, 'memory_used': 408}, '894f403f-b2bc-40f5-a376-10f4f91f2fe0': {'duration': 996, 'init_duration': None, 'memory_size': 2560, 'memory_used': 408}, '8c05838f-bb7f-4b13-a1a7-fea1f952dd6b': {'duration': 4179, 'init_duration': None, 'memory_size': 2560, 'memory_used': 613}, '61726cf2-b358-41ab-ad9a-c0968d6b5e25': {'duration': 39, 'init_duration': None, 'memory_size': 2560, 'memory_used': 613}, '8457e1ec-cab8-4e7a-b1bc-341e1ec7aeb1': {'duration': 21, 'init_duration': None, 'memory_size': 2560, 'memory_used': 613}, '48442b6c-186f-4ef7-bf90-a0943364048f': {'duration': 1104, 'init_duration': None, 'memory_size': 2560, 'memory_used': 613}, '2602e0d5-143a-43dc-95bc-634af077ff89': {'duration': 1362, 'init_duration': None, 'memory_size': 2560, 'memory_used': 613}, '873aad36-cc94-48a7-b459-78b12547d005': {'duration': 2553, 'init_duration': None, 'memory_size': 2560, 'memory_used': 613}, '0fbe3405-dc20-4eb5-83e5-9ab17d834ae2': {'duration': 571, 'init_duration': None, 'memory_size': 2560, 'memory_used': 613}, '67c7562a-d88f-4a8d-b219-3fdc79007c5f': {'duration': 734, 'init_duration': None, 'memory_size': 2560, 'memory_used': 613}, 'fc8ffdea-b967-4954-8a93-f2d210de10ba': {'duration': 3393, 'init_duration': None, 'memory_size': 2560, 'memory_used': 613}, '02468a03-cc2d-40ee-aa88-e310c3b0ba15': {'duration': 1468, 'init_duration': None, 'memory_size': 2560, 'memory_used': 613}, '90b4dd40-43ca-4157-b394-43b312395861': {'duration': 4859, 'init_duration': None, 'memory_size': 2560, 'memory_used': 695}, '25598cbb-3d8e-4bdc-9c89-e13a85b3c991': {'duration': 2192, 'init_duration': None, 'memory_size': 2560, 'memory_used': 695}, 'e08127b1-7b43-478b-879d-783809dcef1b': {'duration': 6906, 'init_duration': None, 'memory_size': 2560, 'memory_used': 839}, 'ae2748ac-823a-44eb-b8f5-4e3c940e6011': {'duration': 9600, 'init_duration': None, 'memory_size': 2560, 'memory_used': 1029}, '931e4744-5fc3-4dbb-bd6d-3c25889108e0': {'duration': 15522, 'init_duration': None, 'memory_size': 2560, 'memory_used': 1371}, 'cb2500cb-a3e4-489d-a3dd-eae83425d210': {'duration': 32560, 'init_duration': None, 'memory_size': 2560, 'memory_used': 2181}, '58afed77-836a-4986-836b-bfabd6636069': {'duration': 5783, 'init_duration': None, 'memory_size': 2560, 'memory_used': 2181}, '48cbefe4-a93e-4ffe-b0ac-0532d190a330': {'duration': 13570, 'init_duration': None, 'memory_size': 2560, 'memory_used': 2181}, '7751bd50-165e-4158-9db3-c5cf7fc0c640': {'duration': 28063, 'init_duration': None, 'memory_size': 2560, 'memory_used': 2181}, 'b1efe930-54f9-4b17-9c3b-5558743bfc95': {'duration': 12046, 'init_duration': None, 'memory_size': 2560, 'memory_used': 2181}, '163e1e75-d2db-4186-9639-7f9a5a55ca7f': {'duration': 17808, 'init_duration': None, 'memory_size': 2560, 'memory_used': 2181}, 'c011b750-2c6e-44e8-8f9f-bb8d2238e533': {'duration': 32866, 'init_duration': None, 'memory_size': 2560, 'memory_used': 2226}, '84ff5160-7d2e-4245-a34d-eab3b4279106': {'duration': 177, 'init_duration': 1234, 'memory_size': 2688, 'memory_used': 163}, 'b5755dd0-8220-42e1-93b5-1cf6b2123ec1': {'duration': 529, 'init_duration': None, 'memory_size': 2688, 'memory_used': 232}, '6f06de69-6cff-4543-99f9-69b62c041b0b': {'duration': 41, 'init_duration': None, 'memory_size': 2688, 'memory_used': 232}, 'dcd34a32-d0bc-41af-a5d5-4dc173b5d8c5': {'duration': 635, 'init_duration': None, 'memory_size': 2688, 'memory_used': 270}, '8c5a54af-b573-489b-aa7f-f1dd79068a70': {'duration': 867, 'init_duration': None, 'memory_size': 2688, 'memory_used': 306}, 'c47644f5-c665-467a-80c3-69631e2a139d': {'duration': 1156, 'init_duration': None, 'memory_size': 2688, 'memory_used': 347}, '798e4db0-d11f-4193-877d-ac06aa776d5a': {'duration': 1946, 'init_duration': None, 'memory_size': 2688, 'memory_used': 438}, '4d5bbe8c-1853-4230-8c07-9442f513843f': {'duration': 2554, 'init_duration': None, 'memory_size': 2688, 'memory_used': 508}, '1f78f389-75d2-41c9-be9f-34e9faa89865': {'duration': 3396, 'init_duration': None, 'memory_size': 2688, 'memory_used': 586}, '14c12207-5d2a-48ff-95aa-80c8b08f7fd7': {'duration': 4302, 'init_duration': None, 'memory_size': 2688, 'memory_used': 672}, 'e5a5c4d6-5867-4a00-afa7-52fadf3a4860': {'duration': 73, 'init_duration': None, 'memory_size': 2688, 'memory_used': 672}, '1882f44f-5ee6-49ee-a4fb-03ed8583fde9': {'duration': 83, 'init_duration': None, 'memory_size': 2688, 'memory_used': 672}, '72c5af01-9ae6-4d6f-941d-b5835f6d8612': {'duration': 1608, 'init_duration': None, 'memory_size': 2688, 'memory_used': 672}, '993f820b-d5c2-466e-87a2-a0d6b95e16ba': {'duration': 954, 'init_duration': None, 'memory_size': 2688, 'memory_used': 672}, 'c2443b84-680f-4310-9dbf-b67ff1b263c6': {'duration': 1753, 'init_duration': None, 'memory_size': 2688, 'memory_used': 672}, 'cfdae3a4-620f-4c92-831e-99e8eb849390': {'duration': 2203, 'init_duration': None, 'memory_size': 2688, 'memory_used': 672}, '0d38f1db-d88a-40a0-b24e-c0a6e50ebcc4': {'duration': 621, 'init_duration': None, 'memory_size': 2688, 'memory_used': 672}, '7912291f-f3c4-4eca-bf19-20191c136df1': {'duration': 2977, 'init_duration': None, 'memory_size': 2688, 'memory_used': 672}, '165a0447-ea0e-4b1f-8501-50dc298faeef': {'duration': 852, 'init_duration': None, 'memory_size': 2688, 'memory_used': 672}, '42d981c0-313c-4683-978e-f943fbbec10f': {'duration': 1132, 'init_duration': None, 'memory_size': 2688, 'memory_used': 672}, '03d6d0bf-3ff9-4253-8e61-aafcff9aa40e': {'duration': 3825, 'init_duration': None, 'memory_size': 2688, 'memory_used': 672}, '0accd56f-0de8-4903-9da2-cf80ad64f165': {'duration': 5181, 'init_duration': None, 'memory_size': 2688, 'memory_used': 727}, '42e30859-61bf-45ff-8efb-c66fbbb4d216': {'duration': 7093, 'init_duration': None, 'memory_size': 2688, 'memory_used': 876}, '63ace721-5e90-4b05-bc8f-4917fa4a098f': {'duration': 12935, 'init_duration': None, 'memory_size': 2688, 'memory_used': 1258}, 'a3d10b2f-e5d5-4fd0-aba5-904c7f51562e': {'duration': 5458, 'init_duration': None, 'memory_size': 2688, 'memory_used': 1258}, 'd72cd202-40f9-4aa8-851c-50713807cd51': {'duration': 31464, 'init_duration': None, 'memory_size': 2688, 'memory_used': 2210}, '6330f72b-b49c-4dd0-9ce4-7782787659a6': {'duration': 5425, 'init_duration': None, 'memory_size': 2688, 'memory_used': 2210}, '878ac42c-d2ee-4504-a88d-1403a21f4cc7': {'duration': 6374, 'init_duration': None, 'memory_size': 2688, 'memory_used': 2210}, '1a6d4535-8e70-46ba-881d-de46f607350b': {'duration': 23227, 'init_duration': None, 'memory_size': 2688, 'memory_used': 2210}, 'a18a06c9-bb73-4e90-8fa0-617a40c61459': {'duration': 30759, 'init_duration': None, 'memory_size': 2688, 'memory_used': 2211}, 'ec14b13c-317c-47bb-9eaf-ffc4d332de0c': {'duration': 6261, 'init_duration': 1166, 'memory_size': 2944, 'memory_used': 844}, 'a1d4685a-273b-4201-80bd-fec769a8c9d0': {'duration': 7354, 'init_duration': None, 'memory_size': 2944, 'memory_used': 952}, 'f9bcdc84-41ff-462d-904d-dc52a0d49340': {'duration': 9804, 'init_duration': None, 'memory_size': 2944, 'memory_used': 1127}, 'f342f071-26e8-4937-bfeb-f9ce1d697861': {'duration': 10436, 'init_duration': None, 'memory_size': 2944, 'memory_used': 1190}, '93054cc8-74a6-44c1-8d49-2da8255e59bc': {'duration': 11426, 'init_duration': None, 'memory_size': 2944, 'memory_used': 1254}, '14f7d77d-2095-460b-952a-262836826319': {'duration': 19782, 'init_duration': None, 'memory_size': 2944, 'memory_used': 1751}, 'a600218b-7a9e-45a4-b2a2-82275c8995d2': {'duration': 24266, 'init_duration': None, 'memory_size': 2944, 'memory_used': 1993}, 'a81adbae-c92f-4044-aea3-452386527d77': {'duration': 24246, 'init_duration': None, 'memory_size': 2944, 'memory_used': 1994}, 'e92e15f6-069c-405b-8080-2fe10d373c57': {'duration': 11, 'init_duration': 1077, 'memory_size': 2048, 'memory_used': 111}, '8b5884f2-7d22-4c67-a3db-e7872f2d5af0': {'duration': 18348, 'init_duration': None, 'memory_size': 2048, 'memory_used': 1354}, 'c7d4dbc5-9010-46e1-865a-7af4fe6425ad': {'duration': 5596, 'init_duration': None, 'memory_size': 2048, 'memory_used': 1354}, 'c78e28c9-100b-4c0a-bac3-79e77c561aad': {'duration': 2668, 'init_duration': None, 'memory_size': 2048, 'memory_used': 2048}, 'dfc9d102-24c3-4e21-9aa1-41f1bb39615a': {'duration': 18265, 'init_duration': None, 'memory_size': 2048, 'memory_used': 1314}, '2211364b-b4b8-439b-8cb5-b51a1cfafb0c': {'duration': 35036, 'init_duration': None, 'memory_size': 2048, 'memory_used': 2048}, '28bb7681-20c6-4545-8afe-538a39bde1c5': {'duration': 28722, 'init_duration': None, 'memory_size': 2048, 'memory_used': 2048}, '2faf8d0e-de63-4ec7-a55a-aeea09ef58d3': {'duration': 4203, 'init_duration': None, 'memory_size': 2048, 'memory_used': 2048}, '87f95921-e179-40f6-a496-f2fcd50eadfe': {'duration': 35222, 'init_duration': None, 'memory_size': 2048, 'memory_used': 2048}, 'ef56b220-b40b-48ee-92e3-c836b23dbc3a': {'duration': 164, 'init_duration': 1203, 'memory_size': 3008, 'memory_used': 160}, 'fc567b7d-921f-4b4c-acf5-10d41307241d': {'duration': 33, 'init_duration': None, 'memory_size': 3008, 'memory_used': 160}, '9c17954b-fee7-4df4-bd66-01ec115edbd3': {'duration': 29, 'init_duration': None, 'memory_size': 3008, 'memory_used': 160}, 'eb9172f5-d7d6-4c8b-8374-45efd817ba5d': {'duration': 676, 'init_duration': None, 'memory_size': 3008, 'memory_used': 266}, 'f55494c9-c62f-4f31-9652-2abb574b164d': {'duration': 80, 'init_duration': None, 'memory_size': 3008, 'memory_used': 266}, 'eefec3af-7632-414d-aae2-5b21cfe89f66': {'duration': 48, 'init_duration': None, 'memory_size': 3008, 'memory_used': 266}, 'aa280c4a-09a5-4e50-913c-2609dd07c13a': {'duration': 726, 'init_duration': None, 'memory_size': 3008, 'memory_used': 292}, 'e386565f-1f15-4ec9-9716-d62fad2c4add': {'duration': 159, 'init_duration': None, 'memory_size': 3008, 'memory_used': 292}, 'c5238b46-6e81-472c-8f8b-838fd8502d5e': {'duration': 114, 'init_duration': None, 'memory_size': 3008, 'memory_used': 292}, '3aa9445f-d9f0-4adb-9575-9cd55c10b5bf': {'duration': 796, 'init_duration': None, 'memory_size': 3008, 'memory_used': 292}, '31894a92-e68d-4db5-bfcf-c94a42680b53': {'duration': 857, 'init_duration': None, 'memory_size': 3008, 'memory_used': 302}, '9373705b-2c6f-4e82-8923-091b7635f173': {'duration': 357, 'init_duration': None, 'memory_size': 3008, 'memory_used': 302}, 'c4074ee7-00d9-44b4-8e01-1e6d1b2535c0': {'duration': 1162, 'init_duration': None, 'memory_size': 3008, 'memory_used': 343}, '247316c6-20f2-40cd-832a-3476b7582139': {'duration': 442, 'init_duration': None, 'memory_size': 3008, 'memory_used': 343}, '7e103d70-a1ba-4b5f-9528-f6cd9ca3886c': {'duration': 1392, 'init_duration': None, 'memory_size': 3008, 'memory_used': 373}, '5b7e061d-40ed-4ff7-a31f-582652cf3274': {'duration': 4, 'init_duration': None, 'memory_size': 3008, 'memory_used': 373}, 'd7fc84c3-a541-4d1a-9caa-92dea280f135': {'duration': 555, 'init_duration': None, 'memory_size': 3008, 'memory_used': 373}, '657148e2-6a8f-490f-88fa-3cea8e52252b': {'duration': 623, 'init_duration': None, 'memory_size': 3008, 'memory_used': 373}, '6219bb61-b499-450e-8dad-2764c683c300': {'duration': 1751, 'init_duration': None, 'memory_size': 3008, 'memory_used': 421}, '7dc89813-e888-4881-95b5-31fd3213b2e9': {'duration': 941, 'init_duration': None, 'memory_size': 3008, 'memory_used': 421}, '12129a89-6684-4d71-8e32-e764a0a41138': {'duration': 3, 'init_duration': None, 'memory_size': 3008, 'memory_used': 421}, '0f92120d-2acc-4a3d-975f-6c62807e9c23': {'duration': 7, 'init_duration': None, 'memory_size': 3008, 'memory_used': 421}, '019fbfe6-8da4-426d-aa7e-7a657f19fe35': {'duration': 9, 'init_duration': None, 'memory_size': 3008, 'memory_used': 421}, '16368773-2a3a-4bb9-9e50-688367fcb72e': {'duration': 3577, 'init_duration': None, 'memory_size': 3008, 'memory_used': 614}, 'f327446e-b481-40b8-8f19-d215a7deb45d': {'duration': 57, 'init_duration': None, 'memory_size': 3008, 'memory_used': 614}, '010c31a9-6aba-4347-b6c0-636c7fc191e7': {'duration': 70, 'init_duration': None, 'memory_size': 3008, 'memory_used': 614}, 'd972f600-30cb-402d-9100-7240834aab23': {'duration': 2002, 'init_duration': None, 'memory_size': 3008, 'memory_used': 614}, 'dfbe1dd5-9d2c-4bcf-a2d3-d5e257e125dd': {'duration': 7922, 'init_duration': None, 'memory_size': 3008, 'memory_used': 1010}, 'fb2c881c-a1d6-4f83-b226-af4e34e8f363': {'duration': 9302, 'init_duration': None, 'memory_size': 3008, 'memory_used': 1128}, '8cb721c0-af62-4c60-9148-31d320bb325f': {'duration': 13416, 'init_duration': None, 'memory_size': 3008, 'memory_used': 1382}, 'b571f498-c805-453a-bd86-07f8c1c8a4c4': {'duration': 7078, 'init_duration': None, 'memory_size': 3008, 'memory_used': 1382}, 'a2873a3c-edce-477b-ac54-105f43c2cb0d': {'duration': 32368, 'init_duration': None, 'memory_size': 3008, 'memory_used': 2414}, 'af5aabe0-1e3a-4813-a32d-5289d0c7a8f0': {'duration': 11974, 'init_duration': None, 'memory_size': 3008, 'memory_used': 2414}, '329078fd-6efc-42cc-a5d5-788a1e91766b': {'duration': 15370, 'init_duration': None, 'memory_size': 3008, 'memory_used': 2414}, 'b0def75b-4e06-4b00-9de9-40ad1c8594ad': {'duration': 11094, 'init_duration': None, 'memory_size': 3008, 'memory_used': 2414}, 'b89ee0a1-494d-498d-b1ba-aba6f02098a9': {'duration': 16274, 'init_duration': None, 'memory_size': 3008, 'memory_used': 2414}, 'fd79eab4-bc39-4391-a2aa-e5aca9af8706': {'duration': 344, 'init_duration': 1221, 'memory_size': 2688, 'memory_used': 190}, 'e93eae2d-d3d9-4ae7-921e-1c4295b3f63f': {'duration': 310, 'init_duration': None, 'memory_size': 2688, 'memory_used': 192}, '1a04df2a-af95-494a-a322-1632450d9583': {'duration': 382, 'init_duration': None, 'memory_size': 2688, 'memory_used': 210}, 'ab612502-84bd-4094-9768-072f91e55a21': {'duration': 430, 'init_duration': None, 'memory_size': 2688, 'memory_used': 220}, 'b0298e5d-18c8-4430-adcf-9057fa7b5c47': {'duration': 1545, 'init_duration': None, 'memory_size': 2688, 'memory_used': 371}, 'abd9aa35-16e3-4578-90f7-78f01d9bd8f1': {'duration': 3, 'init_duration': None, 'memory_size': 2688, 'memory_used': 371}, '8f3817eb-ea8d-4f69-8be6-63a86514254f': {'duration': 2228, 'init_duration': None, 'memory_size': 2688, 'memory_used': 454}, 'ee5e8f4b-4661-47c0-a14c-79085150df3a': {'duration': 2414, 'init_duration': None, 'memory_size': 2688, 'memory_used': 473}, '8e475bd5-7ce2-470f-9210-d04fd4ad834b': {'duration': 846, 'init_duration': None, 'memory_size': 2688, 'memory_used': 473}, 'c860bd25-8b1d-452f-87f0-f7081436dcfc': {'duration': 3922, 'init_duration': None, 'memory_size': 2688, 'memory_used': 610}, '4116bea0-da9e-4541-af50-9d21243f0904': {'duration': 1388, 'init_duration': None, 'memory_size': 2688, 'memory_used': 610}, '4326ade5-0b05-40fe-b56f-a2e664142081': {'duration': 64, 'init_duration': None, 'memory_size': 2688, 'memory_used': 610}, '826154da-b301-4a06-9448-155ccbfcc0ab': {'duration': 1041, 'init_duration': None, 'memory_size': 2688, 'memory_used': 610}, 'e9ae0f29-7218-473e-b57e-6c33f2330129': {'duration': 1264, 'init_duration': None, 'memory_size': 2688, 'memory_used': 610}, 'f7469a0a-bad3-41fe-ab7f-2748fd0b2fb6': {'duration': 2576, 'init_duration': None, 'memory_size': 2688, 'memory_used': 610}, '2ed46c69-8067-48be-9c38-99f62d947dd3': {'duration': 691, 'init_duration': None, 'memory_size': 2688, 'memory_used': 610}, '87ad1d83-9645-4b6b-abcb-87fc918a5df5': {'duration': 3163, 'init_duration': None, 'memory_size': 2688, 'memory_used': 610}, '1dd9da42-6c4a-439b-8f62-09a106252d67': {'duration': 1395, 'init_duration': None, 'memory_size': 2688, 'memory_used': 610}, '80c8f30f-0087-445c-ab66-6607d6a7348c': {'duration': 1635, 'init_duration': None, 'memory_size': 2688, 'memory_used': 610}, '6854b1a3-b03d-4ebf-9081-11799207fb8e': {'duration': 1795, 'init_duration': None, 'memory_size': 2688, 'memory_used': 610}, 'f459c044-78a4-4cd4-bb40-f7e1ad65f206': {'duration': 6143, 'init_duration': None, 'memory_size': 2688, 'memory_used': 819}, '398b2839-4b67-41b9-9ae6-66805dff6e34': {'duration': 2958, 'init_duration': None, 'memory_size': 2688, 'memory_used': 819}, '81fff8b3-8686-49d9-b57f-5b29f1e61405': {'duration': 10631, 'init_duration': None, 'memory_size': 2688, 'memory_used': 1122}, '86031260-3641-43b9-87a4-2bd297b2e4b8': {'duration': 28190, 'init_duration': None, 'memory_size': 2688, 'memory_used': 2058}, '3875c52e-828a-48b8-9961-ef1b3693fcbb': {'duration': 8349, 'init_duration': None, 'memory_size': 2688, 'memory_used': 2058}, 'df47e87c-feff-400c-abf7-8d0766380b8c': {'duration': 21222, 'init_duration': None, 'memory_size': 2688, 'memory_used': 2058}, '5b7b0b29-490d-4798-b2b6-a14462f0de06': {'duration': 28451, 'init_duration': None, 'memory_size': 2688, 'memory_used': 2102}, '0abf7219-3f53-49d3-bc33-0c59a8f62872': {'duration': 33, 'init_duration': 1204, 'memory_size': 2432, 'memory_used': 119}, '6be7df30-f131-475f-971d-efa57e51a7e2': {'duration': 33550, 'init_duration': None, 'memory_size': 2432, 'memory_used': 2117}, 'de4ef26c-f889-45f5-924f-e63b6c98a7e8': {'duration': 7068, 'init_duration': None, 'memory_size': 2432, 'memory_used': 2117}, 'd7bfd434-817c-4e0b-b29b-40287ba2c2e6': {'duration': 10015, 'init_duration': None, 'memory_size': 2432, 'memory_used': 2117}, '2a25683c-a868-44c9-be24-d5b3582918cb': {'duration': 35469, 'init_duration': None, 'memory_size': 2432, 'memory_used': 2249}, '88566a60-b1a9-466d-b202-2fc057e3b19c': {'duration': 26659, 'init_duration': None, 'memory_size': 2432, 'memory_used': 2249}, '0bb953f1-648c-4157-ac38-8144523535ff': {'duration': 31144, 'init_duration': None, 'memory_size': 2432, 'memory_used': 2249}, '9d03e838-cf3a-44ce-87aa-a3c8017615c8': {'duration': 23443, 'init_duration': 1221, 'memory_size': 2688, 'memory_used': 1792}, 'dc5b46e7-8ab2-4f55-8bb1-7da4a959b05a': {'duration': 11489, 'init_duration': None, 'memory_size': 2688, 'memory_used': 1792}, 'ba812020-6c14-4c36-91f1-872c8b6014d0': {'duration': 13280, 'init_duration': None, 'memory_size': 2688, 'memory_used': 1792}, '9f6cd7f8-8a5e-4664-9a4e-c40426e99251': {'duration': 16801, 'init_duration': None, 'memory_size': 2688, 'memory_used': 1792}, '97cfb923-6127-44da-b101-76138a07ad93': {'duration': 32289, 'init_duration': None, 'memory_size': 2688, 'memory_used': 2252}, 'a71f6f35-ce78-4269-aaaa-cf5e9504a99c': {'duration': 26602, 'init_duration': None, 'memory_size': 2688, 'memory_used': 2252}, 'ed4a2acd-5d55-4600-90a1-7adb08c2037c': {'duration': 18918, 'init_duration': None, 'memory_size': 2688, 'memory_used': 2252}, 'db396beb-019c-44bd-8905-6f870a841c43': {'duration': 257, 'init_duration': 1220, 'memory_size': 2432, 'memory_used': 175}, '56cc6617-a931-4978-ba28-fe39b29a4c12': {'duration': 6697, 'init_duration': None, 'memory_size': 2432, 'memory_used': 771}, '45d6a440-9836-42f1-92f5-7fbcc49cc7f9': {'duration': 7502, 'init_duration': None, 'memory_size': 2432, 'memory_used': 847}, 'e1c0d1a7-e5db-4d21-be72-e847c783781a': {'duration': 17739, 'init_duration': None, 'memory_size': 2432, 'memory_used': 1422}, '238b932f-2ba5-4a01-aebf-1fb6ef9ba6cc': {'duration': 9604, 'init_duration': None, 'memory_size': 2432, 'memory_used': 1422}, '1206a44e-1c70-450b-8456-8f1b7fb3f4e0': {'duration': 10010, 'init_duration': None, 'memory_size': 2432, 'memory_used': 1422}, 'a8cacd0f-213c-4bf6-9edf-95da9c9132cd': {'duration': 8762, 'init_duration': None, 'memory_size': 2432, 'memory_used': 1422}, 'ae115883-d36f-4bfa-8756-7a894d26e2cb': {'duration': 33174, 'init_duration': None, 'memory_size': 2432, 'memory_used': 2120}, 'd18d9517-a2ca-4937-8748-864a65f5c412': {'duration': 24095, 'init_duration': None, 'memory_size': 2432, 'memory_used': 2120}, '9fea8035-6d94-4279-b350-d88d8a4c456f': {'duration': 18592, 'init_duration': None, 'memory_size': 2432, 'memory_used': 2120}, '734db36b-67ef-49f4-9dcf-b2867b357e58': {'duration': 20529, 'init_duration': None, 'memory_size': 2432, 'memory_used': 2120}, 'bf5ba046-1f4b-4a73-b52e-360cd3c34213': {'duration': 13, 'init_duration': 1210, 'memory_size': 2560, 'memory_used': 111}, 'f9247db3-8f88-4349-b82a-2b248a05c6ed': {'duration': 5139, 'init_duration': None, 'memory_size': 2560, 'memory_used': 676}, 'd9b1b17b-7a27-4f2c-ad68-47eb00c5bce9': {'duration': 25257, 'init_duration': None, 'memory_size': 2560, 'memory_used': 1831}, 'c3131c5a-4901-481c-96d7-f872e8f1919a': {'duration': 30183, 'init_duration': None, 'memory_size': 2560, 'memory_used': 2078}, '468a9ce5-40d3-49fe-9ef3-95eb18fd3730': {'duration': 15387, 'init_duration': None, 'memory_size': 2560, 'memory_used': 2078}, '79dc8283-70c2-41aa-b7e2-c56e2cfa51a6': {'duration': 22920, 'init_duration': None, 'memory_size': 2560, 'memory_used': 2078}, '9f31f619-1b75-417d-adab-fbcd5d18de29': {'duration': 19886, 'init_duration': None, 'memory_size': 2560, 'memory_used': 2078}, '3e9fff51-2801-4607-b091-ff8d934e0efb': {'duration': 12, 'init_duration': 1221, 'memory_size': 2816, 'memory_used': 114}, 'e2b4df04-da36-47a7-9bd5-a58a1d430b4d': {'duration': 381, 'init_duration': None, 'memory_size': 2816, 'memory_used': 203}, 'ce35dfe0-ea73-4f6d-9ef6-4dd9e005b63f': {'duration': 15862, 'init_duration': None, 'memory_size': 2816, 'memory_used': 1424}, 'e16e0a5d-c03d-4220-8fc6-3e627cc34cfe': {'duration': 25763, 'init_duration': None, 'memory_size': 2816, 'memory_used': 1996}, '10bdc576-b795-4991-906e-7f2677592b53': {'duration': 29234, 'init_duration': None, 'memory_size': 2816, 'memory_used': 2166}, '5066de9c-67ae-4a5a-8a71-5504811810e8': {'duration': 28819, 'init_duration': None, 'memory_size': 2816, 'memory_used': 2166}, '0e45e0fb-5a8b-40eb-b16a-3128bf27be20': {'duration': 22080, 'init_duration': 1176, 'memory_size': 2816, 'memory_used': 1790}, 'b7b2b2bb-3dd7-46b1-9460-e18842406670': {'duration': 23548, 'init_duration': None, 'memory_size': 2816, 'memory_used': 1911}, '044c44c3-7920-44dd-b59d-37d614a864ab': {'duration': 21245, 'init_duration': None, 'memory_size': 2816, 'memory_used': 1911}, '78d05dda-bf1b-47e6-83a3-a12e40655334': {'duration': 32229, 'init_duration': None, 'memory_size': 2816, 'memory_used': 2339}, '9d2fdf26-1ae2-453c-beb3-685d99cd359b': {'duration': 12, 'init_duration': 1130, 'memory_size': 1792, 'memory_used': 112}, '7e06ef59-e98b-4059-8778-eb63b221574c': {'duration': 358, 'init_duration': None, 'memory_size': 1792, 'memory_used': 175}, '10861fb7-bd70-47a8-91c5-b9b1646d4bf4': {'duration': 580, 'init_duration': None, 'memory_size': 1792, 'memory_used': 210}, '9ec5c82c-7e72-431b-a9b0-1a0609e0cca2': {'duration': 407, 'init_duration': None, 'memory_size': 1792, 'memory_used': 210}, 'd066bb7a-31f6-44b2-9b25-9c1c4df1400b': {'duration': 2194, 'init_duration': None, 'memory_size': 1792, 'memory_used': 371}, '9fcc01be-442a-4a76-b9c8-bac78f1e7ebe': {'duration': 3231, 'init_duration': None, 'memory_size': 1792, 'memory_used': 454}, 'b33118c6-6bdc-4a9a-b64d-5c5a99405cc9': {'duration': 6274, 'init_duration': None, 'memory_size': 1792, 'memory_used': 653}, '20fe5387-5ec2-4c92-a144-ee8f21d89f02': {'duration': 171, 'init_duration': None, 'memory_size': 1792, 'memory_used': 653}, '2f4649e6-5be0-4fe3-b1dc-185564690abe': {'duration': 196, 'init_duration': None, 'memory_size': 1792, 'memory_used': 653}, 'c7b0d91e-22b8-4cbe-abf6-7bd2419e0e2f': {'duration': 3098, 'init_duration': None, 'memory_size': 1792, 'memory_used': 653}, '98c8ad5f-80a9-4f00-87ab-6930c94f08cf': {'duration': 598, 'init_duration': None, 'memory_size': 1792, 'memory_used': 653}, '681fbba2-d628-4660-aacd-2ad34f695c7e': {'duration': 4068, 'init_duration': None, 'memory_size': 1792, 'memory_used': 653}, '013d85f5-ce45-41ef-8d86-c315685d15c0': {'duration': 5172, 'init_duration': None, 'memory_size': 1792, 'memory_used': 653}, 'a925342a-ac38-4d38-95cf-043c1170b597': {'duration': 18190, 'init_duration': None, 'memory_size': 1792, 'memory_used': 1269}, 'aaddfeb5-c5c1-43fc-824c-9f49428beaad': {'duration': 22801, 'init_duration': None, 'memory_size': 1792, 'memory_used': 1467}, '474f8542-3b85-4b05-b0f2-960a9a1dca80': {'duration': 6575, 'init_duration': None, 'memory_size': 1792, 'memory_used': 1467}, 'e0d4096c-802c-40e0-90f8-34527a7f1032': {'duration': 10772, 'init_duration': None, 'memory_size': 1792, 'memory_used': 1467}, '15697f49-58c9-4d14-9e0b-7d1b5d6afb75': {'duration': 9672, 'init_duration': None, 'memory_size': 1792, 'memory_used': 1467}, 'e062c9f4-9d85-40dd-a4d1-c47a39fb0926': {'duration': 15534, 'init_duration': None, 'memory_size': 1792, 'memory_used': 1467}, '20e367f7-4700-4b4c-b38b-1191e77a2bb8': {'duration': 14194, 'init_duration': None, 'memory_size': 1792, 'memory_used': 1467}, '6dc11d43-050f-462e-8737-cf758d4eda86': {'duration': 21033, 'init_duration': None, 'memory_size': 1792, 'memory_used': 1467}, '5d5e5666-a00c-4603-837a-3eda17c36402': {'duration': 4424, 'init_duration': None, 'memory_size': 1792, 'memory_used': 1792}, '4811034a-b893-44b3-8869-6367f48588a7': {'duration': 5309, 'init_duration': None, 'memory_size': 1792, 'memory_used': 1792}, '1624cb92-4b1b-45ce-a78c-74fe57f8e688': {'duration': 33697, 'init_duration': None, 'memory_size': 1792, 'memory_used': 1781}, '3cacd889-b192-495e-bfc4-b99ec61bf9a6': {'duration': 4347, 'init_duration': None, 'memory_size': 1792, 'memory_used': 1792}, '03d0ee86-d1e7-4773-8558-0097ef607935': {'duration': 5548, 'init_duration': None, 'memory_size': 1792, 'memory_used': 1792}, '599bb00c-b5fd-4b5c-9937-f2a1eacc5726': {'duration': 29, 'init_duration': 1193, 'memory_size': 2560, 'memory_used': 123}, '7f7110fa-a8fd-4372-b460-d6d2d5692778': {'duration': 4829, 'init_duration': None, 'memory_size': 2560, 'memory_used': 655}, '17267d13-fa98-486d-9276-a1482d751104': {'duration': 21292, 'init_duration': None, 'memory_size': 2560, 'memory_used': 1639}, '2759fb76-61c0-49e5-b84e-963afe7e7c34': {'duration': 12010, 'init_duration': None, 'memory_size': 2560, 'memory_used': 1639}, '22087fcf-2b71-4486-967c-b74e3881f00d': {'duration': 25540, 'init_duration': None, 'memory_size': 2560, 'memory_used': 1873}, 'f7d7da61-282c-461f-86c4-3f629f848bb1': {'duration': 19934, 'init_duration': None, 'memory_size': 2560, 'memory_used': 1873}, '46a8c294-18f6-46b5-97e5-0e0694479d7c': {'duration': 14130, 'init_duration': None, 'memory_size': 2560, 'memory_used': 1873}, 'dfc4262f-1d0c-4439-91c3-e26ce90bf597': {'duration': 16015, 'init_duration': None, 'memory_size': 2560, 'memory_used': 1873}, '56d42124-b46e-401e-b92a-5e2d68caa67e': {'duration': 36830, 'init_duration': None, 'memory_size': 2560, 'memory_used': 2387}, '666f93d4-68c2-4e19-a876-0529e76f87c5': {'duration': 35754, 'init_duration': None, 'memory_size': 2560, 'memory_used': 2387}, '266c1505-8fc7-46e2-86ff-99a22f8bdc19': {'duration': 34, 'init_duration': 1235, 'memory_size': 1920, 'memory_used': 124}, '86455ea5-4b32-4177-8ba4-7141269cd9f2': {'duration': 8764, 'init_duration': None, 'memory_size': 1920, 'memory_used': 823}, 'f462b205-3e88-409a-bd5b-71bff7ae32c2': {'duration': 4774, 'init_duration': None, 'memory_size': 1920, 'memory_used': 823}, '81e2cf0e-4dd3-4e47-8e99-3d59a86c3165': {'duration': 25972, 'init_duration': None, 'memory_size': 1920, 'memory_used': 1603}, '79386c0a-0143-4308-a714-99d3541eb356': {'duration': 11641, 'init_duration': None, 'memory_size': 1920, 'memory_used': 1603}, '19ba6fd2-ddf0-4075-a6ad-d0505787123c': {'duration': 14299, 'init_duration': None, 'memory_size': 1920, 'memory_used': 1603}, '8a56f1d9-0cd1-46e3-8f0f-22855cbdf5fb': {'duration': 9933, 'init_duration': None, 'memory_size': 1920, 'memory_used': 1603}, '43d313a5-32a0-44e4-8a4b-1a379019aaa3': {'duration': 25933, 'init_duration': None, 'memory_size': 1920, 'memory_used': 1604}, 'd45b71c6-4d03-4de2-b677-cde7e67e3092': {'duration': 35702, 'init_duration': None, 'memory_size': 1920, 'memory_used': 1920}, '4b212277-8efd-4f30-9cbb-2903ccd14766': {'duration': 4419, 'init_duration': None, 'memory_size': 1920, 'memory_used': 1920}, '9c911d14-8609-4cf4-9d54-719f3a6e8002': {'duration': 149, 'init_duration': 1327, 'memory_size': 1664, 'memory_used': 145}, 'f389b172-aafd-49c6-bbd3-01094a6a346f': {'duration': 781, 'init_duration': None, 'memory_size': 1664, 'memory_used': 222}, 'c800483d-b17e-4e1c-b00b-6b7e9ee0420a': {'duration': 990, 'init_duration': None, 'memory_size': 1664, 'memory_used': 256}, '119f8b92-d10e-4f12-8bb0-ba3c3c16a21f': {'duration': 38, 'init_duration': None, 'memory_size': 1664, 'memory_used': 256}, '09ebf3de-a535-465b-b547-8e42a0ba8f63': {'duration': 53, 'init_duration': None, 'memory_size': 1664, 'memory_used': 256}, 'de9153d2-91b9-48e0-868f-9889302916eb': {'duration': 1368, 'init_duration': None, 'memory_size': 1664, 'memory_used': 305}, 'fd2502a8-aff8-458c-a741-b7b8c8f35762': {'duration': 344, 'init_duration': None, 'memory_size': 1664, 'memory_used': 305}, '24c04e83-7a1c-44cf-b935-7c42fd36b095': {'duration': 2263, 'init_duration': None, 'memory_size': 1664, 'memory_used': 375}, 'db3caa8c-9061-4b5c-85cf-855f7c6f86bd': {'duration': 2882, 'init_duration': None, 'memory_size': 1664, 'memory_used': 423}, 'b59c596b-07a4-47f8-a6e9-c0460f247f5d': {'duration': 3, 'init_duration': None, 'memory_size': 1664, 'memory_used': 423}, 'f1f6ba48-bf7d-4510-a79c-266247d45bac': {'duration': 20165, 'init_duration': None, 'memory_size': 1664, 'memory_used': 1241}, 'a7fa851b-434d-4b36-9384-625bbe217bb9': {'duration': 27538, 'init_duration': None, 'memory_size': 1664, 'memory_used': 1510}, '27b39cdd-0fd4-4799-8738-c9fd84873b71': {'duration': 4586, 'init_duration': None, 'memory_size': 1664, 'memory_used': 1664}, '6174857d-027f-4f02-887f-ae10420e2ce0': {'duration': 16436, 'init_duration': None, 'memory_size': 1664, 'memory_used': 1031}, '42e61688-3401-436e-ad5c-93d9ef79b122': {'duration': 24294, 'init_duration': None, 'memory_size': 1664, 'memory_used': 1379}, '60d2442f-2f4c-4eb9-8dac-2ecc4a4c313e': {'duration': 13173, 'init_duration': None, 'memory_size': 1664, 'memory_used': 1379}, '0e3d9de3-595d-467a-8994-3ad5015cf07f': {'duration': 2436, 'init_duration': None, 'memory_size': 1664, 'memory_used': 1664}, '151fdf1f-88ce-4847-b23f-1e7cd3cd9b22': {'duration': 21695, 'init_duration': None, 'memory_size': 1664, 'memory_used': 1245}, 'a3bfddd1-644a-4d08-a99e-87a777e2501e': {'duration': 25365, 'init_duration': None, 'memory_size': 1664, 'memory_used': 1412}, '7f6c793c-62c4-4691-a0c7-13a2dcedeb2b': {'duration': 2382, 'init_duration': None, 'memory_size': 1664, 'memory_used': 1664}, 'ca7171d1-8518-4138-8db1-e1cd383beb4c': {'duration': 31224, 'init_duration': None, 'memory_size': 1664, 'memory_used': 1592}, 'a7885f77-2904-47eb-bf37-b604646c2ef6': {'duration': 4529, 'init_duration': None, 'memory_size': 1664, 'memory_used': 1664}, '38d3b8fb-e00a-4ac5-b3a7-bae784a831fc': {'duration': 117, 'init_duration': 1203, 'memory_size': 3008, 'memory_used': 148}, 'd9dd5424-4a99-421c-8d4b-59f5f71e1ec3': {'duration': 762, 'init_duration': None, 'memory_size': 3008, 'memory_used': 275}, '6e7924ba-bce3-4f4d-a40f-a798d6c39f4f': {'duration': 788, 'init_duration': None, 'memory_size': 3008, 'memory_used': 288}, 'a1571436-c2bd-4905-8e7e-32aef9fba894': {'duration': 2801, 'init_duration': None, 'memory_size': 3008, 'memory_used': 529}, 'bb98bd8b-9c50-44f5-a427-684aa60f1192': {'duration': 1145, 'init_duration': None, 'memory_size': 3008, 'memory_used': 529}, '44424b87-46f3-4bf3-8bfe-b700ed8182f5': {'duration': 3721, 'init_duration': None, 'memory_size': 3008, 'memory_used': 633}, '952a371e-6523-46b2-9862-34cf880464e3': {'duration': 95, 'init_duration': None, 'memory_size': 3008, 'memory_used': 633}, 'efeca3d3-b016-4bd0-b23e-c70920ce3f84': {'duration': 139, 'init_duration': None, 'memory_size': 3008, 'memory_used': 633}, '93fb8ae6-7ef7-4d83-a294-5d1701471ec5': {'duration': 186, 'init_duration': None, 'memory_size': 3008, 'memory_used': 633}, '949dc6ce-9b98-4349-89ea-fe485589b681': {'duration': 300, 'init_duration': None, 'memory_size': 3008, 'memory_used': 633}, 'c6e94667-865d-46a5-8655-a75aa4fbdfd6': {'duration': 948, 'init_duration': None, 'memory_size': 3008, 'memory_used': 633}, 'd118f995-2568-4fa6-b095-2cca0715b7a8': {'duration': 2007, 'init_duration': None, 'memory_size': 3008, 'memory_used': 633}, 'c70210aa-24db-4c4c-8608-b905e7c4dc91': {'duration': 568, 'init_duration': None, 'memory_size': 3008, 'memory_used': 633}, 'ed736ba8-13f7-4834-b484-596837beb5d3': {'duration': 2696, 'init_duration': None, 'memory_size': 3008, 'memory_used': 633}, '633926d2-d42b-4017-9d7c-73ba5a3e5a2e': {'duration': 779, 'init_duration': None, 'memory_size': 3008, 'memory_used': 633}, '5962408f-ddd8-4c0d-9144-228be3238058': {'duration': 1032, 'init_duration': None, 'memory_size': 3008, 'memory_used': 633}, '1d6f6762-f43a-446c-8d43-72baf7575186': {'duration': 3452, 'init_duration': None, 'memory_size': 3008, 'memory_used': 633}, '335ad094-90f5-4f08-9e57-08d83164859c': {'duration': 3710, 'init_duration': None, 'memory_size': 3008, 'memory_used': 634}, 'a05c1476-2cbf-4700-98dd-f5ab7e57a859': {'duration': 1603, 'init_duration': None, 'memory_size': 3008, 'memory_used': 634}, '5c36929f-905e-46f7-8dbb-71b0820670e7': {'duration': 2510, 'init_duration': None, 'memory_size': 3008, 'memory_used': 634}, '64fe1fd3-bf2e-482e-a207-a81de9538090': {'duration': 10436, 'init_duration': None, 'memory_size': 3008, 'memory_used': 1160}, '62499245-e992-43f1-8721-60b4ddb85d4a': {'duration': 11576, 'init_duration': None, 'memory_size': 3008, 'memory_used': 1255}, 'a85ac045-49d4-4789-83a4-504ddc488536': {'duration': 18829, 'init_duration': None, 'memory_size': 3008, 'memory_used': 1676}, '92e128d5-c3d8-4bae-afef-78993eb84417': {'duration': 8625, 'init_duration': None, 'memory_size': 3008, 'memory_used': 1676}, 'f7607ba9-a8b3-4aa0-a0de-352cef9c2bf5': {'duration': 9315, 'init_duration': None, 'memory_size': 3008, 'memory_used': 1676}, '70156c6c-a9cd-4d30-a7fc-02035429eb45': {'duration': 6735, 'init_duration': None, 'memory_size': 3008, 'memory_used': 1676}, '7c9edbb5-8368-4290-b048-2d73dfb83cec': {'duration': 7827, 'init_duration': None, 'memory_size': 3008, 'memory_used': 1676}, '1e0284e0-c9a2-4b0c-9b42-80661a46f903': {'duration': 21254, 'init_duration': None, 'memory_size': 3008, 'memory_used': 1832}, 'ab39d55b-5067-4650-ae84-bb30d566a439': {'duration': 10296, 'init_duration': None, 'memory_size': 3008, 'memory_used': 1832}, 'ca80bf59-e692-496e-8f7f-9b2ca851690e': {'duration': 29256, 'init_duration': None, 'memory_size': 3008, 'memory_used': 2251}, '5c5a5980-3b3c-43f5-b976-2c7d7aa97a36': {'duration': 22481, 'init_duration': None, 'memory_size': 3008, 'memory_used': 2251}, '28ac31fa-d147-4d55-91b3-b81b8021654a': {'duration': 2239, 'init_duration': 1266, 'memory_size': 1408, 'memory_used': 330}, 'eb531a81-dacf-448e-a483-c7ee1bba78c1': {'duration': 3655, 'init_duration': None, 'memory_size': 1408, 'memory_used': 423}, '38188e0f-3235-4e00-abe5-b0fec91ffa6f': {'duration': 2985, 'init_duration': None, 'memory_size': 1408, 'memory_used': 423}, '33112c01-10b2-4458-b53e-5d0a19786874': {'duration': 7272, 'init_duration': None, 'memory_size': 1408, 'memory_used': 613}, 'de74723e-b18b-4ed0-8e48-e2c9c2fa106e': {'duration': 9608, 'init_duration': None, 'memory_size': 1408, 'memory_used': 726}, '84cbce7b-2c9e-4b12-8d9c-49f5dc29f4f3': {'duration': 3169, 'init_duration': None, 'memory_size': 1408, 'memory_used': 1408}, 'ab392276-113a-44c8-895f-6e2aac61c5de': {'duration': 16004, 'init_duration': None, 'memory_size': 1408, 'memory_used': 920}, '36072f34-cb36-490e-b62a-10985103b581': {'duration': 4526, 'init_duration': None, 'memory_size': 1408, 'memory_used': 1408}, '48fef17c-fcd6-49da-85e3-e65a44e86baa': {'duration': 23333, 'init_duration': None, 'memory_size': 1408, 'memory_used': 1181}, '535ade42-9cb8-45f7-9af7-a876d76e3703': {'duration': 27206, 'init_duration': None, 'memory_size': 1408, 'memory_used': 1344}, '1bba1df6-a2a5-4a86-a239-5f364352979e': {'duration': 4234, 'init_duration': None, 'memory_size': 1408, 'memory_used': 1408}, '95800fd2-f273-45c4-b45d-75ad0cb2480a': {'duration': 5790, 'init_duration': None, 'memory_size': 1408, 'memory_used': 1408}, '4f19f6e9-3520-42f5-b180-acb2465d77fc': {'duration': 6103, 'init_duration': None, 'memory_size': 1408, 'memory_used': 1408}, '57602be8-0d56-4395-a76d-f255e3c34467': {'duration': 4054, 'init_duration': None, 'memory_size': 1408, 'memory_used': 1408}, '9c73355f-023e-49e6-b7ce-022e8e90e7a7': {'duration': 6058, 'init_duration': None, 'memory_size': 1408, 'memory_used': 1408}, '73af471b-4e5a-4867-b3d0-4e8ddb53bd9f': {'duration': 4044, 'init_duration': None, 'memory_size': 1408, 'memory_used': 1408}, 'a3380420-e414-48f0-bf39-262bb6fcf500': {'duration': 4344, 'init_duration': None, 'memory_size': 1408, 'memory_used': 1408}, '3a8c586a-2cdb-4850-85d3-42849a231f37': {'duration': 194, 'init_duration': 1065, 'memory_size': 2560, 'memory_used': 169}, '67159746-403f-46b3-af95-91dd6e9e97a4': {'duration': 378, 'init_duration': None, 'memory_size': 2560, 'memory_used': 203}, 'ea4aaa60-a06c-4b21-9b7b-3a70911f7a88': {'duration': 535, 'init_duration': None, 'memory_size': 2560, 'memory_used': 242}, '5829b606-e2a9-4410-82fc-700240449522': {'duration': 596, 'init_duration': None, 'memory_size': 2560, 'memory_used': 255}, '23092d7c-5a23-424f-8dc5-7e9f8065e91b': {'duration': 1920, 'init_duration': None, 'memory_size': 2560, 'memory_used': 421}, '62516e08-b14e-4aa4-aaa6-9a492df816b9': {'duration': 727, 'init_duration': None, 'memory_size': 2560, 'memory_used': 421}, 'bb9832c9-1506-48de-ae5e-494ec4ecf16f': {'duration': 2870, 'init_duration': None, 'memory_size': 2560, 'memory_used': 531}, 'e7444279-2341-4821-96d7-f535764292a0': {'duration': 4154, 'init_duration': None, 'memory_size': 2560, 'memory_used': 656}, 'edb3f41c-3b0e-4517-b3b6-9f2291419c68': {'duration': 15, 'init_duration': None, 'memory_size': 2560, 'memory_used': 656}, '56fa0953-9a3d-45b7-98f4-d5ac2efc5763': {'duration': 1425, 'init_duration': None, 'memory_size': 2560, 'memory_used': 656}, 'a0edc11b-c85a-4167-bb48-4d39521429fc': {'duration': 104, 'init_duration': None, 'memory_size': 2560, 'memory_used': 656}, 'a0e6c834-2da7-4ceb-9e39-2b9cc9eae2d1': {'duration': 125, 'init_duration': None, 'memory_size': 2560, 'memory_used': 656}, 'f7d6ca70-a643-4f09-837c-5aba2803f62f': {'duration': 1666, 'init_duration': None, 'memory_size': 2560, 'memory_used': 656}, '639a82d8-3424-4442-a072-dd86f975604e': {'duration': 308, 'init_duration': None, 'memory_size': 2560, 'memory_used': 656}, 'ba20c407-ca86-4103-b5a0-ea5ce6353e1e': {'duration': 2130, 'init_duration': None, 'memory_size': 2560, 'memory_used': 656}, 'c9668be7-e311-4f76-a1bc-2478b482699e': {'duration': 425, 'init_duration': None, 'memory_size': 2560, 'memory_used': 656}, 'a1ed5b4f-8e29-404e-be69-1acaf42a795b': {'duration': 455, 'init_duration': None, 'memory_size': 2560, 'memory_used': 656}, '79f0eac2-b4cc-492c-86b1-6febba57875a': {'duration': 2779, 'init_duration': None, 'memory_size': 2560, 'memory_used': 656}, '9a0613cf-5ec6-4c74-8ad8-6328d2111143': {'duration': 793, 'init_duration': None, 'memory_size': 2560, 'memory_used': 656}, '516bcb03-2a6e-41c1-a33b-edc7a1b2f575': {'duration': 1046, 'init_duration': None, 'memory_size': 2560, 'memory_used': 656}, '1e7a31ea-36cb-436d-959d-d388c089ba80': {'duration': 4553, 'init_duration': None, 'memory_size': 2560, 'memory_used': 702}, '0f219879-e301-4b2e-8eda-6b436a48a5f0': {'duration': 2089, 'init_duration': None, 'memory_size': 2560, 'memory_used': 702}, '6bab2ce0-bee4-471c-925b-1e586bd2fd4f': {'duration': 4065, 'init_duration': None, 'memory_size': 2560, 'memory_used': 702}, '076785a8-3c4d-46a2-9541-3354ef0155b5': {'duration': 2396, 'init_duration': None, 'memory_size': 2560, 'memory_used': 702}, '1133d986-3d63-4edd-a6c3-838e27fcb05c': {'duration': 2225, 'init_duration': None, 'memory_size': 2560, 'memory_used': 702}, 'e5f705bb-410e-453a-a32b-4599759df09b': {'duration': 2772, 'init_duration': None, 'memory_size': 2560, 'memory_used': 702}, 'b503542f-4e0f-468b-a608-53ea923c166b': {'duration': 9948, 'init_duration': None, 'memory_size': 2560, 'memory_used': 1100}, '116d0d94-bd4d-4619-9a9a-265cabf879a2': {'duration': 22951, 'init_duration': None, 'memory_size': 2560, 'memory_used': 1873}, '8a27ef77-1856-46a5-9624-f8930204cafc': {'duration': 7233, 'init_duration': None, 'memory_size': 2560, 'memory_used': 1873}, '6982fc48-2864-46f3-a882-cce7197f996f': {'duration': 8362, 'init_duration': None, 'memory_size': 2560, 'memory_used': 1873}, '6f2a9361-451c-4523-bea4-eea2504fe89f': {'duration': 9192, 'init_duration': None, 'memory_size': 2560, 'memory_used': 1873}, 'fd22f9fe-eb0c-40c9-9c07-549e77df65ff': {'duration': 21006, 'init_duration': None, 'memory_size': 2560, 'memory_used': 1873}, '728d67bd-4703-4139-ae2f-71dafc7a1466': {'duration': 6313, 'init_duration': 1088, 'memory_size': 640, 'memory_used': 368}, '2d65f030-f9d3-4e5e-ae1c-34e4d2aa7430': {'duration': 1827, 'init_duration': None, 'memory_size': 640, 'memory_used': 368}, '82a0e099-bda7-4089-aaff-fcd6c72eca63': {'duration': 5186, 'init_duration': None, 'memory_size': 640, 'memory_used': 640}, 'e394ba7d-839a-4732-a54f-bafbb73e9e94': {'duration': 13917, 'init_duration': None, 'memory_size': 640, 'memory_used': 482}, '91f008a6-fa0e-4f44-9179-be68cabb76ca': {'duration': 2935, 'init_duration': None, 'memory_size': 640, 'memory_used': 640}, 'd1595e76-548f-4624-b9d0-20cd39af3f5f': {'duration': 16851, 'init_duration': None, 'memory_size': 640, 'memory_used': 561}, 'd477e8c0-b7c8-4e13-9cfa-89dc77829fef': {'duration': 2984, 'init_duration': None, 'memory_size': 640, 'memory_used': 640}, '0973fac2-f095-459f-8e7c-44105b059501': {'duration': 7778, 'init_duration': None, 'memory_size': 640, 'memory_used': 640}, '239b6729-dbc5-4113-b6ea-21590a114263': {'duration': 6172, 'init_duration': None, 'memory_size': 640, 'memory_used': 640}, '4eca6325-d89e-4a0f-a49c-b5f03de2cd9c': {'duration': 6106, 'init_duration': None, 'memory_size': 640, 'memory_used': 640}, '3408de1a-93a8-49c1-afe6-8579dc0e40ff': {'duration': 7621, 'init_duration': None, 'memory_size': 640, 'memory_used': 640}, '81e4784c-ba53-4b20-b467-1494dc7fd916': {'duration': 7457, 'init_duration': None, 'memory_size': 640, 'memory_used': 640}, '28cf716b-6958-428b-be05-b8cea233e303': {'duration': 7901, 'init_duration': None, 'memory_size': 640, 'memory_used': 640}, 'a8f249ce-e42c-4c17-a13b-7d2bed125876': {'duration': 8046, 'init_duration': None, 'memory_size': 640, 'memory_used': 640}, 'aad4397b-ed2a-41ea-9793-9cf1b3c15fe7': {'duration': 5981, 'init_duration': None, 'memory_size': 640, 'memory_used': 640}, 'c9222a19-b2df-4405-bc91-8c464b14b756': {'duration': 6096, 'init_duration': None, 'memory_size': 640, 'memory_used': 640}, '7de51219-25fd-44f5-920e-a25c2edff0c7': {'duration': 7884, 'init_duration': None, 'memory_size': 640, 'memory_used': 640}, '65fb6ab6-3bff-4869-b7f1-cd234029f631': {'duration': 7728, 'init_duration': None, 'memory_size': 640, 'memory_used': 640}, '92d88abc-dffb-4cf1-8a16-17001b3921ac': {'duration': 5930, 'init_duration': None, 'memory_size': 640, 'memory_used': 640}, 'a5819aa8-940d-4709-8477-e546952343c8': {'duration': 5932, 'init_duration': None, 'memory_size': 640, 'memory_used': 640}, '3b48b6cb-4b7e-4df0-91ac-12e43322fc80': {'duration': 7871, 'init_duration': None, 'memory_size': 640, 'memory_used': 640}, '0e516638-31f1-40e4-be78-5b211a5ba265': {'duration': 7752, 'init_duration': None, 'memory_size': 640, 'memory_used': 640}, '03ad9e32-8399-48e0-8488-6bd19e8aa403': {'duration': 3986, 'init_duration': 1312, 'memory_size': 768, 'memory_used': 312}, 'cf593b68-95dc-4b3d-b1bc-3e75f21d0469': {'duration': 363, 'init_duration': None, 'memory_size': 768, 'memory_used': 312}, 'd9ca6fed-2ae8-432e-87ba-a6ba2e8bfc56': {'duration': 5290, 'init_duration': None, 'memory_size': 768, 'memory_used': 768}, 'bb21d70a-3b49-4c54-9811-5af6cc52eec5': {'duration': 7646, 'init_duration': None, 'memory_size': 768, 'memory_used': 768}, 'ae3623c2-24d8-4861-b2a3-1178647b20ce': {'duration': 5718, 'init_duration': None, 'memory_size': 768, 'memory_used': 768}, '9af3a52a-cec2-462d-8190-79d2ddbacc92': {'duration': 5629, 'init_duration': None, 'memory_size': 768, 'memory_used': 768}, 'f31c5671-bdae-453d-9a79-fd49a5c4d516': {'duration': 22609, 'init_duration': None, 'memory_size': 768, 'memory_used': 737}, '667c356b-76ac-4649-abc4-c4d714c8e79e': {'duration': 4527, 'init_duration': None, 'memory_size': 768, 'memory_used': 768}, 'c07c4fbb-0e1a-49e2-8b36-28a84824a066': {'duration': 7539, 'init_duration': None, 'memory_size': 768, 'memory_used': 768}, '870f9bf1-cc4d-48bf-a157-75d33245174a': {'duration': 23350, 'init_duration': None, 'memory_size': 768, 'memory_used': 763}, '144f461d-1d13-4dbb-9c8d-612cefb47f86': {'duration': 3561, 'init_duration': None, 'memory_size': 768, 'memory_used': 768}, 'be08ec62-ad9f-4ac7-8d58-d4346541aa49': {'duration': 7604, 'init_duration': None, 'memory_size': 768, 'memory_used': 768}, '1f982b6f-1903-4f75-9a13-dc0be65ef2c1': {'duration': 5725, 'init_duration': None, 'memory_size': 768, 'memory_used': 768}, '24a7c64d-1492-451f-a11d-6e58dca0da74': {'duration': 7539, 'init_duration': None, 'memory_size': 768, 'memory_used': 768}, 'b59c5227-a5c9-47e7-81f9-d66ef9411d66': {'duration': 5651, 'init_duration': None, 'memory_size': 768, 'memory_used': 768}, '5cb75d3a-9aba-46f1-9812-75458a309807': {'duration': 7585, 'init_duration': None, 'memory_size': 768, 'memory_used': 768}, 'dafbd6a3-5495-43ec-852a-1dd938583372': {'duration': 5652, 'init_duration': None, 'memory_size': 768, 'memory_used': 768}, '6dc80913-5e92-442f-898b-0d049a1743d1': {'duration': 7569, 'init_duration': None, 'memory_size': 768, 'memory_used': 768}, 'f5b5821e-2fa1-4e18-93cc-6483ec45e2f3': {'duration': 7532, 'init_duration': None, 'memory_size': 768, 'memory_used': 768}, '7b5fe653-4f0e-49ef-942a-0792aa623077': {'duration': 7623, 'init_duration': None, 'memory_size': 768, 'memory_used': 768}, '02edf626-5520-4dde-8143-df66d6fc3df8': {'duration': 6267, 'init_duration': None, 'memory_size': 768, 'memory_used': 768}, '837c2b84-ab1c-4422-a9d0-3ccc3dc0c89e': {'duration': 5572, 'init_duration': None, 'memory_size': 768, 'memory_used': 768}, '7572232c-7206-4402-9419-76d314b99697': {'duration': 7955, 'init_duration': None, 'memory_size': 768, 'memory_used': 768}, 'f7d54ca6-dc0e-4f76-ad8c-d9af14070bde': {'duration': 7560, 'init_duration': None, 'memory_size': 768, 'memory_used': 768}, '8497f041-af0a-45b9-822b-27680a9e6fc8': {'duration': 50, 'init_duration': 1060, 'memory_size': 2816, 'memory_used': 131}, '3c4b3100-b763-46e9-ac14-a9f4dd3c6d64': {'duration': 19983, 'init_duration': None, 'memory_size': 2816, 'memory_used': 1754}, '79b01104-d61e-4a67-931e-92078006c58e': {'duration': 29986, 'init_duration': None, 'memory_size': 2816, 'memory_used': 2342}, 'f78fe0d8-d728-4408-91be-6bd81d78f1f3': {'duration': 6306, 'init_duration': None, 'memory_size': 2816, 'memory_used': 2342}, '62229ee9-ac24-4e59-a4d5-2b0bc10d8581': {'duration': 14886, 'init_duration': None, 'memory_size': 2816, 'memory_used': 2342}, '9cdbaca4-ad6e-4b97-89c3-7c59738f4543': {'duration': 27527, 'init_duration': None, 'memory_size': 2816, 'memory_used': 2342}, 'f26cb29d-1ca3-4cfd-bc9e-9885779b70cb': {'duration': 22091, 'init_duration': None, 'memory_size': 2816, 'memory_used': 2342}, '007947fc-c807-4795-966b-682ce0246f8b': {'duration': 23, 'init_duration': 1346, 'memory_size': 2816, 'memory_used': 122}, '6b9ff70f-f027-4b94-a06c-8dee902a9682': {'duration': 5905, 'init_duration': None, 'memory_size': 2816, 'memory_used': 774}, '88e897d2-85c3-4831-be21-6b17fbcfff56': {'duration': 18697, 'init_duration': None, 'memory_size': 2816, 'memory_used': 1604}, 'f325cd36-678b-42aa-8224-2ce3f7418609': {'duration': 8719, 'init_duration': None, 'memory_size': 2816, 'memory_used': 1604}, '7dfb7265-9b11-4681-8b39-3ccffcef7b4e': {'duration': 4912, 'init_duration': None, 'memory_size': 2816, 'memory_used': 1604}, '09966b6b-dc20-44c0-8022-bffe0bfe6970': {'duration': 11346, 'init_duration': None, 'memory_size': 2816, 'memory_used': 1604}, 'ff886d42-a18e-48bf-bbb6-a821d0826120': {'duration': 7115, 'init_duration': None, 'memory_size': 2816, 'memory_used': 1604}, '9b4316bf-c0ae-4dde-aea4-3e469f239ab4': {'duration': 8276, 'init_duration': None, 'memory_size': 2816, 'memory_used': 1604}, 'e22bf0c1-0a05-41b0-8c16-0c932ceb1705': {'duration': 22613, 'init_duration': None, 'memory_size': 2816, 'memory_used': 1835}, '93da2bcc-a56f-4729-8830-d10d0734c1ee': {'duration': 10859, 'init_duration': None, 'memory_size': 2816, 'memory_used': 1835}, 'f8e952ce-3cf4-4e3e-a4ba-7a39862096a7': {'duration': 13934, 'init_duration': None, 'memory_size': 2816, 'memory_used': 1835}, 'cb1f0c35-02d9-49b0-82c1-1db62cc2efdc': {'duration': 13297, 'init_duration': None, 'memory_size': 2816, 'memory_used': 1835}, '815ec9bb-73f4-4f68-ba40-ca6c298ccef1': {'duration': 23124, 'init_duration': None, 'memory_size': 2816, 'memory_used': 1876}, '48e62963-5a5b-4316-8211-fbd465e79a94': {'duration': 16, 'init_duration': 1245, 'memory_size': 2688, 'memory_used': 110}, '64469f47-9ba7-4720-ab32-b288d73a7b12': {'duration': 4581, 'init_duration': None, 'memory_size': 2688, 'memory_used': 652}, '12dc3e3c-8de8-46b2-b956-669196322d6d': {'duration': 18537, 'init_duration': None, 'memory_size': 2688, 'memory_used': 1563}, 'ae1fa908-bf86-493e-bff1-7aebd74c0f97': {'duration': 25180, 'init_duration': None, 'memory_size': 2688, 'memory_used': 1910}, '3eed9555-630b-4c17-ae34-dfed6fab90ca': {'duration': 13326, 'init_duration': None, 'memory_size': 2688, 'memory_used': 1910}, '6b4c43f0-5710-4d2c-9c7a-956f708ba53f': {'duration': 19615, 'init_duration': None, 'memory_size': 2688, 'memory_used': 1910}, '823618f4-38cf-494f-a007-22087b49f275': {'duration': 23988, 'init_duration': None, 'memory_size': 2688, 'memory_used': 1910}, '578f944d-0fd9-4e11-9a43-99fbd97fab86': {'duration': 31, 'init_duration': 1274, 'memory_size': 896, 'memory_used': 113}, '7a6cbfb7-3571-4e2c-afdb-f126fffa9b1e': {'duration': 10967, 'init_duration': None, 'memory_size': 896, 'memory_used': 570}, '3062e1e0-a3ee-4bd4-92a4-38f1a2190c6f': {'duration': 3135, 'init_duration': None, 'memory_size': 896, 'memory_used': 570}, 'fb5f083a-95ce-490b-b4d0-0cda3ca46752': {'duration': 6432, 'init_duration': None, 'memory_size': 896, 'memory_used': 570}, '23fb986d-3a91-4928-9417-d2ed3d994d5c': {'duration': 3811, 'init_duration': None, 'memory_size': 896, 'memory_used': 570}, '2871431b-bf4b-4640-b635-d3534e8edfa8': {'duration': 20513, 'init_duration': None, 'memory_size': 896, 'memory_used': 848}, 'd9ac18bd-a547-42b7-a8b9-db0e789ae02c': {'duration': 4800, 'init_duration': None, 'memory_size': 896, 'memory_used': 896}, '00291dab-94da-43c0-b2f6-c3415e192915': {'duration': 7046, 'init_duration': None, 'memory_size': 896, 'memory_used': 896}, '1853180e-50f4-4bc6-8602-9dd5705c970a': {'duration': 7028, 'init_duration': None, 'memory_size': 896, 'memory_used': 896}, 'be06888d-a019-4b02-b43e-6c5f9fb64a66': {'duration': 22727, 'init_duration': None, 'memory_size': 896, 'memory_used': 838}, '30cda5df-c5d6-4878-8c3f-2a8330bf19db': {'duration': 2758, 'init_duration': None, 'memory_size': 896, 'memory_used': 896}, 'd68e23c6-ced9-47d9-8cb1-06ece74543b7': {'duration': 20727, 'init_duration': None, 'memory_size': 896, 'memory_used': 789}, 'b4177c55-50c6-4158-b8aa-efd132eafb12': {'duration': 4397, 'init_duration': None, 'memory_size': 896, 'memory_used': 896}, 'ff54b2e6-e287-40b0-8a52-3facde51d48a': {'duration': 6993, 'init_duration': None, 'memory_size': 896, 'memory_used': 896}, 'e0d554d9-3a10-406a-9f45-6ac1c52bf454': {'duration': 5456, 'init_duration': None, 'memory_size': 896, 'memory_used': 896}, '3349201a-b9cf-4805-8e13-70062ae40436': {'duration': 7067, 'init_duration': None, 'memory_size': 896, 'memory_used': 896}, '3fcb16f1-8b32-4e23-9114-f28836752eda': {'duration': 6990, 'init_duration': None, 'memory_size': 896, 'memory_used': 896}, '09e86d76-f32b-4f25-9ecb-05510b553638': {'duration': 7112, 'init_duration': None, 'memory_size': 896, 'memory_used': 896}, '2cab6599-8f9d-477e-a014-3d1ebb0e7850': {'duration': 5077, 'init_duration': None, 'memory_size': 896, 'memory_used': 896}, 'a8009bc0-32e4-4344-8f3b-6fd4429b9bc7': {'duration': 7005, 'init_duration': None, 'memory_size': 896, 'memory_used': 896}, '5b4f464f-d303-45cd-a99f-688425c5e1de': {'duration': 7081, 'init_duration': None, 'memory_size': 896, 'memory_used': 896}, '230253f6-02af-4174-ab0f-f10fdd55dbfb': {'duration': 5667, 'init_duration': None, 'memory_size': 896, 'memory_used': 896}, 'c221bfaf-b868-486f-a361-8b1b0069c099': {'duration': 7005, 'init_duration': None, 'memory_size': 896, 'memory_used': 896}, '98983c10-f054-46ff-814c-d262b6f03fdf': {'duration': 5481, 'init_duration': None, 'memory_size': 896, 'memory_used': 896}, '1aa30f75-a56e-415c-ab6c-e6b5a6705fcd': {'duration': 88, 'init_duration': 1070, 'memory_size': 3008, 'memory_used': 144}, 'b6aae0bf-7366-4d30-99e7-34eacc933220': {'duration': 3, 'init_duration': None, 'memory_size': 3008, 'memory_used': 144}, '47c11133-64fd-47ff-a3c8-1058e034f3ed': {'duration': 6, 'init_duration': None, 'memory_size': 3008, 'memory_used': 144}, '8a3c2617-50f3-4838-85c6-36fc58044603': {'duration': 4, 'init_duration': None, 'memory_size': 3008, 'memory_used': 144}, '364e6a63-bc1f-46fa-9fd9-5453aca4aa23': {'duration': 16, 'init_duration': None, 'memory_size': 3008, 'memory_used': 144}, 'ee01cbc5-8c42-4576-9ecf-b83f61e1eaf5': {'duration': 1762, 'init_duration': None, 'memory_size': 3008, 'memory_used': 430}, '347f8890-5c16-4bb0-9594-2e52cffdde1b': {'duration': 5035, 'init_duration': None, 'memory_size': 3008, 'memory_used': 781}, 'c7b19577-9e32-4022-a9d9-8f4d0c9b7795': {'duration': 5709, 'init_duration': None, 'memory_size': 3008, 'memory_used': 858}, '9eb82ece-b8ee-4d13-8463-74f6aacde9c5': {'duration': 5927, 'init_duration': None, 'memory_size': 3008, 'memory_used': 884}, 'e3c083eb-7bde-4dc8-b7a1-f001940478f7': {'duration': 7922, 'init_duration': None, 'memory_size': 3008, 'memory_used': 1050}, 'cb6de1f4-3c75-4d2d-af89-1874c6251bf0': {'duration': 9950, 'init_duration': None, 'memory_size': 3008, 'memory_used': 1201}, '4c8182e6-1f93-4e87-be26-caee7b3e0533': {'duration': 11200, 'init_duration': None, 'memory_size': 3008, 'memory_used': 1297}, '8e678bf0-27f2-4520-a511-80517dd51d34': {'duration': 20383, 'init_duration': None, 'memory_size': 3008, 'memory_used': 1881}, '83c74172-f1c0-4a40-b346-9cb942ff744f': {'duration': 8260, 'init_duration': None, 'memory_size': 3008, 'memory_used': 1881}, 'ddf7ace6-d894-455e-b01a-025014fd786e': {'duration': 21465, 'init_duration': None, 'memory_size': 3008, 'memory_used': 1962}, 'd7a286f6-c4e9-4c72-b956-fc8089804539': {'duration': 16074, 'init_duration': None, 'memory_size': 3008, 'memory_used': 1962}, '0bb36272-7a27-44f5-90c8-bd98b95131ca': {'duration': 16676, 'init_duration': None, 'memory_size': 3008, 'memory_used': 1962}, '59e06021-7d35-419f-ab8c-5c4905a5445d': {'duration': 37257, 'init_duration': 1266, 'memory_size': 2048, 'memory_used': 2036}, '65ad11c7-a169-4f7d-bb18-872099b80dd1': {'duration': 6249, 'init_duration': None, 'memory_size': 2048, 'memory_used': 2036}, 'bd9bdd48-81bf-4e0e-bced-a071bf96996e': {'duration': 17333, 'init_duration': None, 'memory_size': 2048, 'memory_used': 2036}, '16dab940-d238-4388-8e11-8860fd6b4b41': {'duration': 10763, 'init_duration': None, 'memory_size': 2048, 'memory_used': 2036}, 'b574dfcc-b574-4cc8-8d1e-935bb62052ca': {'duration': 14229, 'init_duration': None, 'memory_size': 2048, 'memory_used': 2036}, 'a1f20134-11ef-407d-8b6d-e61eee80bec8': {'duration': 32427, 'init_duration': None, 'memory_size': 2048, 'memory_used': 2036}, '221cc4a5-3545-456d-a6ff-0d32fc8fc793': {'duration': 4497, 'init_duration': None, 'memory_size': 2048, 'memory_used': 2048}, '960047c7-e3ff-49c2-85e1-5a769fb8a650': {'duration': 5648, 'init_duration': None, 'memory_size': 2048, 'memory_used': 2048}}
'''

In [2]:
import boto3
import csv
from boto3.dynamodb.conditions import Attr

def download_all_dynamodb_records(table_name):
    dynamodb = boto3.resource('dynamodb', region_name='ap-southeast-2')
    table = dynamodb.Table(table_name)
    
    all_records = []
    column_name = 'start_time'
    last_evaluated_key = None

    while True:
        if last_evaluated_key:
            response = table.scan(
                FilterExpression=Attr(column_name).exists(),
                ExclusiveStartKey=last_evaluated_key
            )
        else:
            response = table.scan(
                FilterExpression=Attr(column_name).exists()
            )

        all_records.extend(response['Items'])

        last_evaluated_key = response.get('LastEvaluatedKey')
        if not last_evaluated_key:
            break

    return all_records

# Example usage
table_name = 'sebs-graph-mst_logs'
all_records = download_all_dynamodb_records(table_name)

# # # Print all records
# for record in all_records:
#     print(record)



# Specify the file path and name
csv_file = 'sebs-graph-mst.csv'

# Get all unique keys from all records
all_keys = set()
for record in all_records:
    all_keys.update(record.keys())

# Write the records to the CSV file
with open(csv_file, 'w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=all_keys)
    writer.writeheader()
    for record in all_records:
        # Fill missing values with an empty string
        for key in all_keys:
            if key not in record:
                record[key] = ''
        writer.writerow(record)

print(f"Records have been written to {csv_file}")

Records have been written to sebs-graph-mst.csv


In [3]:
import pandas as pd

# Read the CSV file into a DataFrame
df = pd.read_csv('float_records_Aug2.csv')

# Remove rows where 'duration' column value is NaN
df.dropna(subset=['version'], inplace=True)

# Write the updated DataFrame back to the CSV file
df.to_csv('float_records_Aug2.csv', index=False)

In [13]:
import pandas as pd

df = pd.read_csv('linpack_records_July30.csv')
df

,version,total_memory,tmp_max,cpu_user_time,fd_use,tmp_used,rx_bytes,request_id,payload,insights_duration,...,memory_used,memory_size,memory_utilisation,shutdown_reason,cpu_system_time,start_time,tmp_free,function_name,tx_bytes,used_memory_max
0,RAM2432,2432.0,550461440.0,460.0,21.0,12140544.0,0.0,edeebc6d-97b8-4ec5-84c6-133191403f58,{'n': 2810},638.0,...,239,2432,9.0,0,60.0,1722327457117,538320896.0,workbench-linpack,0.0,239.0
1,RAM2688,2688.0,550461440.0,370.0,22.0,12140544.0,31140.0,95d79bc6-f29a-43f8-b883-0471b4a8e425,{'n': 2210},363.0,...,183,2688,6.0,0,170.0,1722327454873,538320896.0,workbench-linpack,19235.0,183.0
2,RAM2688,2688.0,550461440.0,1430.0,22.0,12140544.0,15711.0,81059bf8-7d2a-4fe8-9a51-298330ba5e8e,{'n': 4010},1390.0,...,357,2688,13.0,0,330.0,1722327463628,538320896.0,workbench-linpack,11201.0,357.0
3,RAM2176,2176.0,550461440.0,1810.0,21.0,12140544.0,0.0,564dade9-95bf-4bd3-83f9-abef71129de2,{'n': 4210},1780.0,...,520,2176,23.0,0,230.0,1722327495385,538320896.0,workbench-linpack,0.0,520.0
4,RAM1280,1280.0,550461440.0,3640.0,21.0,12140544.0,264.0,fa21af30-a5b6-47a3-8e26-03108c540479,{'n': 5610},6933.0,...,588,1280,45.0,0,340.0,1722327523093,538320896.0,workbench-linpack,120.0,588.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7036,RAM1408,1408.0,550461440.0,640.0,22.0,12140544.0,264.0,2869feb2-22c5-47c9-aa9a-eee7e298a6bf,{'n': 2810},920.0,...,370,1408,26.0,0,90.0,1722327462522,538320896.0,workbench-linpack,120.0,370.0
7037,RAM640,640.0,550461440.0,5020.0,21.0,12140544.0,306.0,81a0ea38-feca-4b11-875b-7efcf4aef0ed,{'n': 6210},18525.0,...,640,640,100.0,0,380.0,1722327546905,538320896.0,workbench-linpack,162.0,640.0
7038,RAM2432,2432.0,550461440.0,6790.0,21.0,12140544.0,7821.0,e4daf2de-cfd0-44f4-ad5c-803158332e78,{'n': 6710},5760.0,...,1392,2432,57.0,0,650.0,1722327569955,538320896.0,workbench-linpack,7905.0,1392.0
7039,RAM2432,2432.0,550461440.0,50.0,21.0,12140544.0,0.0,345c9a34-22ac-414c-b8d2-6923bb1c0f61,{'n': 1010},49.0,...,141,2432,5.0,0,30.0,1722327455453,538320896.0,workbench-linpack,0.0,141.0


In [9]:

import boto3
import time
def cloudwatch_client_from_arn(lambda_arn):
    region = lambda_arn.split(":")[3]
    return boto3.client('logs', region_name=region)


lambda_arn = "arn:aws:lambda:ap-southeast-2:030103857128:function:workbench-matmul"
last_event_time = None
insight_function_name = 'getLogInsights' # ENTER THE LogInsight FUNCTION NAME
executor_function_name = 'executeFunctions' # ENTER THE EXECUTOR FUNCTION NAME

# TODO: Fetch the last streamed log request id
end_time = int(time.time() * 1000)  # Current time in milliseconds
# start_time = end_time - 900000  # 15 minutes ago in milliseconds
start_time = end_time - 16200000
# Create a CloudWatch Logs client
client = cloudwatch_client_from_arn(lambda_arn)


# Get the list of log groups
log_groups = client.describe_log_groups()
log_group_names = [group['logGroupName'] for group 
                    in log_groups['logGroups'] 
                    if group['logGroupName'].endswith('/aws/lambda/' \
                                                        + lambda_arn.split(':')[-1])]

# Get the list of log streams
log_streams = []
for log_group_name in log_group_names:
    next_token = None
    while True:
        if next_token:
            response = client.describe_log_streams(logGroupName=log_group_name,
                                                        orderBy='LastEventTime',
                                                    descending=True,
                                                    nextToken=next_token)
        else:
            response = client.describe_log_streams(logGroupName=log_group_name,
                                                    orderBy='LastEventTime',
                                                    descending=True)
        log_streams += response['logStreams']
        next_token = response.get('nextToken', None)
        if not next_token:
            break

print(len(log_streams))

# Get the list of log events
log_events = []
for log_stream in log_streams:
    next_token = None
    while True:
        if next_token:
            response = client.get_log_events(
                logGroupName=log_group_name,
                logStreamName=log_stream['logStreamName'],
                startTime=start_time,
                endTime=end_time,
                nextToken=next_token
            )
        else:
            response = client.get_log_events(
                logGroupName=log_group_name,
                logStreamName=log_stream['logStreamName'],
                startTime=start_time,
                endTime=end_time
            )
        log_events += response['events']
        last_token = next_token
        next_token = response.get('nextForwardToken', None)
        print(next_token)
        if not next_token or next_token == last_token:
            break

# Parse the log events
parsed_events = {}
print(len(log_events))

# Get the payload value from the executor lambda function
filter_pattern = 'PAYLOAD'
log_events = []
next_token = None

while True:
    if next_token:
        response = client.filter_log_events(
            logGroupName=f'/aws/lambda/{executor_function_name}', # ENTER THE EXECUTOR FUNCTION LOG GROUP
            startTime=start_time,
            endTime=end_time,
            filterPattern=filter_pattern,
            nextToken=next_token
        )
    else:
        response = client.filter_log_events(
            logGroupName=f'/aws/lambda/{executor_function_name}', # ENTER THE EXECUTOR FUNCTION LOG GROUP
            startTime=start_time,
            endTime=end_time,
            filterPattern=filter_pattern
        )
    log_events += response['events']
    next_token = response.get('nextToken', None)
    if not next_token:
        break

print(len(log_events))
# print(parsed_events)




12177
...


KeyboardInterrupt: 

In [1]:
import utils
import boto3
lambda_arn = "arn:aws:lambda:ap-southeast-2:030103857128:function:workbench-pyaes"
power_value = [
    128,
    256,
    384,
    512,
    640,
    768,
    896,
    1024,
    1152,
    1280,
    1408,
    1536,
    1664,
    1792,
    1920,
    2048,
    2176,
    2304,
    2432,
    2560,
    2688,
    2816,
    2944,
    3008
]
lambda_client = utils.lambda_client_from_arn(lambda_arn)
for al in power_value:
    alias = f'RAM{al}' # using Map item selector

    params = {
    'FunctionName': lambda_arn,
    'Name': alias,
    }
    

    try:
        response = lambda_client.get_alias(**params)
        # check if it exists and fetch version ID
        function_version = response['FunctionVersion']
        print(function_version)
        if function_version:
            print(function_version)
            # function_version = function_version['FunctionVersion']
            # delete both alias and version (could be done in parallel!)
            try:
                utils.delete_lambda_alias(lambda_arn, alias)
            except Exception as error:
                print(error)
                continue
            try:
                utils.delete_lambda_version(lambda_arn, function_version)
            except Exception as error:
                print(error)
                continue
        else:
            continue
    except Exception as error:
        if error.response['Error']['Code'] == 'ResourceNotFoundException':
            print('OK, even if version/alias was not found')
            print(error)
        else:
            print(error)
            raise error

3
3
Deleting alias  RAM128
Deleting version  3
4
4
Deleting alias  RAM256
Deleting version  4
5
5
Deleting alias  RAM384
Deleting version  5
6
6
Deleting alias  RAM512
Deleting version  6
7
7
Deleting alias  RAM640
Deleting version  7
8
8
Deleting alias  RAM768
Deleting version  8
9
9
Deleting alias  RAM896
Deleting version  9
10
10
Deleting alias  RAM1024
Deleting version  10
11
11
Deleting alias  RAM1152
Deleting version  11
12
12
Deleting alias  RAM1280
Deleting version  12
13
13
Deleting alias  RAM1408
Deleting version  13
14
14
Deleting alias  RAM1536
Deleting version  14
15
15
Deleting alias  RAM1664
Deleting version  15
16
16
Deleting alias  RAM1792
Deleting version  16
17
17
Deleting alias  RAM1920
Deleting version  17
18
18
Deleting alias  RAM2048
Deleting version  18
19
19
Deleting alias  RAM2176
Deleting version  19
20
20
Deleting alias  RAM2304
Deleting version  20
21
21
Deleting alias  RAM2432
Deleting version  21
22
22
Deleting alias  RAM2560
Deleting version  22
23
23
De

In [1]:
from botocore import config

lambda_config = config.Config(
    read_timeout=900,
    connect_timeout=900,
    retries={"max_attempts": 0}
)

def lambda_client_from_arn(lambda_arn, lambda_config):
    region = lambda_arn.split(":")[3]
    return boto3.client('lambda', region_name=region, config=lambda_config)

In [2]:
import boto3
lambda_ = boto3.client('lambda', region_name='ap-southeast-2', config=lambda_config)
lambda_.list_functions()

{'ResponseMetadata': {'RequestId': 'c6a17e04-0e95-4921-8c10-7c1e422c5bc3',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Tue, 13 Aug 2024 01:24:12 GMT',
   'content-type': 'application/json',
   'content-length': '37492',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'c6a17e04-0e95-4921-8c10-7c1e422c5bc3'},
  'RetryAttempts': 0},
 'Functions': [{'FunctionName': 'logParsingTester',
   'FunctionArn': 'arn:aws:lambda:ap-southeast-2:030103857128:function:logParsingTester',
   'Runtime': 'python3.12',
   'Role': 'arn:aws:iam::030103857128:role/service-role/logParsingTester-role-2dz3yhjz',
   'Handler': 'lambda_function.lambda_handler',
   'CodeSize': 2124,
   'Description': '',
   'Timeout': 900,
   'MemorySize': 128,
   'LastModified': '2024-08-08T03:08:17.000+0000',
   'CodeSha256': '6oRrbVz8ssUK9a8qLsB6k7/hOgRfFzK7iLussitraLQ=',
   'Version': '$LATEST',
   'TracingConfig': {'Mode': 'PassThrough'},
   'RevisionId': '3b0874ec-8b5d-47d7-aacc-574a1df0e8fb',
   'PackageType': 'Z

In [3]:
list = []
length = len([i for i in range(10, 10000, 100)])
for i in range(10, 10000, 100):
    list.append({"payload": {'n': i}, "weight": (1/length)})
list

sum_of_weights = sum(item['weight'] for item in list)
sum_of_weights

1.0

In [3]:
def list_jpg_objects(bucket, prefix):
    s3 = boto3.client('s3')
    jpg_objects = []
    paginator = s3.get_paginator('list_objects_v2')
    pages = paginator.paginate(Bucket=bucket, Prefix=prefix)

    for page in pages:
        if 'Contents' in page:
            for obj in page['Contents']:
                if obj['Key'].endswith('.jpg'):
                    jpg_objects.append(obj['Key'])
    return jpg_objects


In [5]:
list_jpg_objects('image-resizer-upload', '')

['image1.jpg',
 'image2.jpg',
 'image3.jpg',
 'image4.jpg',
 'image5.jpg',
 'image6.jpg',
 'image9.jpg']

In [2]:
# !pip install igraph

In [2]:

# GRAPH PAGERANK
import datetime
import igraph

size = 50000

graph_generating_begin = datetime.datetime.now()
graph = igraph.Graph.Barabasi(size, 100)
graph_generating_end = datetime.datetime.now()

process_begin = datetime.datetime.now()
result = graph.pagerank()
process_end = datetime.datetime.now()

graph_generating_time = (graph_generating_end - graph_generating_begin) / datetime.timedelta(microseconds=1)
process_time = (process_end - process_begin) / datetime.timedelta(microseconds=1)

resp = {
        'result': result[0],
        'measurement': {
            'graph_generating_time': graph_generating_time,
            'compute_time': process_time
        }
}

resp

{'result': 0.00022771219085413827,
 'measurement': {'graph_generating_time': 3537279.0, 'compute_time': 994419.0}}

In [2]:
# GRAPH MST
import datetime
import igraph
size = 10

graph_generating_begin = datetime.datetime.now()
graph = igraph.Graph.Barabasi(size, 1000)
graph_generating_end = datetime.datetime.now()

process_begin = datetime.datetime.now()
result = graph.spanning_tree(None, False)
process_end = datetime.datetime.now()

graph_generating_time = (graph_generating_end - graph_generating_begin) / datetime.timedelta(microseconds=1)
process_time = (process_end - process_begin) / datetime.timedelta(microseconds=1)

resp = {
        'result': result[0],
        'measurement': {
            'graph_generating_time': graph_generating_time,
            'compute_time': process_time
        }
}

resp

{'result': 0,
 'measurement': {'graph_generating_time': 270.0, 'compute_time': 89.0}}

In [11]:
# GRAPH BFS

import datetime
import igraph

size = 10000

graph_generating_begin = datetime.datetime.now()
graph = igraph.Graph.Barabasi(size, 1000)
graph_generating_end = datetime.datetime.now()

process_begin = datetime.datetime.now()
result = graph.bfs(0)
process_end = datetime.datetime.now()

graph_generating_time = (graph_generating_end - graph_generating_begin) / datetime.timedelta(microseconds=1)
process_time = (process_end - process_begin) / datetime.timedelta(microseconds=1)

resp = {
        'result': result,
        'measurement': {
            'graph_generating_time': graph_generating_time,
            'compute_time': process_time
        }
}

resp

{'result': ([0,
   1,
   2,
   3,
   4,
   5,
   6,
   7,
   8,
   9,
   10,
   11,
   12,
   13,
   14,
   15,
   16,
   17,
   18,
   19,
   20,
   21,
   22,
   23,
   24,
   25,
   26,
   27,
   28,
   29,
   30,
   31,
   32,
   33,
   34,
   35,
   36,
   37,
   38,
   39,
   40,
   41,
   42,
   43,
   44,
   45,
   46,
   47,
   48,
   49,
   50,
   51,
   52,
   53,
   54,
   55,
   56,
   57,
   58,
   59,
   60,
   61,
   62,
   63,
   64,
   65,
   66,
   67,
   68,
   69,
   70,
   71,
   72,
   73,
   74,
   75,
   76,
   77,
   78,
   79,
   80,
   81,
   82,
   83,
   84,
   85,
   86,
   87,
   88,
   89,
   90,
   91,
   92,
   93,
   94,
   95,
   96,
   97,
   98,
   99,
   100,
   101,
   102,
   103,
   104,
   105,
   106,
   107,
   108,
   109,
   110,
   111,
   112,
   113,
   114,
   115,
   116,
   117,
   118,
   119,
   120,
   121,
   122,
   123,
   124,
   125,
   126,
   127,
   128,
   129,
   130,
   131,
   132,
   133,
   134,
   135,
   136,
   1

In [8]:
from datetime import datetime                                                   
from random import sample  
from os import path
from time import time                                                           
import os

from jinja2 import Template

SCRIPT_DIR = path.abspath(path.join(path.dirname(os.path.abspath(__name__))))

def handler(event):

    # start timing
    name = event.get('username')
    size = event.get('random_len')
    cur_time = datetime.now()
    random_numbers = sample(range(0, 1000000), size)
    template = Template( open(path.join(SCRIPT_DIR, 'templates', 'template.html'), 'r').read())
    html = template.render(username = name, cur_time = cur_time, random_numbers = random_numbers)
    # end timing
    # dump stats 
    return {'result': html}
handler({'username': 'John', 'random_len': 10})

{'result': '<!DOCTYPE html>\n<html>\n  <head>\n    <title>Randomly generated data.</title>\n    <meta name="viewport" content="width=device-width, initial-scale=1.0">\n    <link href="http://netdna.bootstrapcdn.com/bootstrap/3.0.0/css/bootstrap.min.css" rel="stylesheet" media="screen">\n    <style type="text/css">\n      .container {\n        max-width: 500px;\n        padding-top: 100px;\n      }\n    </style>\n  </head>\n  <body>\n    <div class="container">\n      <p>Welcome John!</p>\n      <p>Data generated at: 2024-08-13 08:02:42.896751!</p>\n      <p>Requested random numbers:</p>\n      <ul>\n        \n        <li>299709</li>\n        \n        <li>838149</li>\n        \n        <li>873896</li>\n        \n        <li>515660</li>\n        \n        <li>642768</li>\n        \n        <li>218045</li>\n        \n        <li>725515</li>\n        \n        <li>397514</li>\n        \n        <li>683056</li>\n        \n        <li>982997</li>\n        \n      </ul>\n    </div>\n  </body

In [2]:
# !pip install pdf2image

In [1]:
import json as json
import boto3
import uuid
import tempfile
# from urllib.parse import unquote_plus
from pdf2image import convert_from_path, convert_from_bytes
from pdf2image.exceptions import (
    PDFInfoNotInstalledError,
    PDFPageCountError,
    PDFSyntaxError
)

In [2]:
images = convert_from_bytes(open('/home/ubuntu/memFigLessDIR/aws-lambda-power-tuning/similarity-lambda-workflow/offline-step/ColdStart_Keynote_Paper.pdf', 'rb').read())

In [5]:
import json
results_file_path = "/home/ubuntu/memFigLessDIR/Parrotfish-UBC/results.json"
with open(results_file_path, 'r') as file:
    results = json.load(file)

In [6]:
results

[]

In [7]:
from boto3.dynamodb.conditions import Attr
import boto3
import pandas as pd
def download_all_dynamodb_records(table_name):
    dynamodb = boto3.resource('dynamodb', region_name='ap-southeast-2')
    table = dynamodb.Table(table_name)
    
    all_records = []
    column_name = 'start_time'
    last_evaluated_key = None

    while True:
        if last_evaluated_key:
            response = table.scan(
                FilterExpression=Attr(column_name).exists(),
                ExclusiveStartKey=last_evaluated_key
            )
        else:
            response = table.scan(
                FilterExpression=Attr(column_name).exists()
            )

        all_records.extend(response['Items'])

        last_evaluated_key = response.get('LastEvaluatedKey')
        if not last_evaluated_key:
            break

    return all_records


# Example usage
function_name = 'sebs-graph-mst'
# Extract function name from lambda ARN
# function_name = function_name.split(':')[-1]
# Define the table name
table_name = f'{function_name}_logs'
all_records = download_all_dynamodb_records(table_name)
df = pd.DataFrame(all_records)

if function_name == 'workbench-matmul' or \
    function_name == 'workbench-linpack' or \
    function_name == 'sebs-floatOperation':
    # Define the regex pattern
    pattern = r"\{'n': (\d+)\}"
    # Replace the column value with the matched group
    df['payload'] = df['payload'].str.replace(pattern, r"\1", regex=True)
    df['payload'] = df['payload'].astype(float)

elif function_name == 'workbench-pyaes':
    pattern = r"\{'length_of_message': (\d+), 'num_of_iterations': (\d+)\}"
    df['payload'] = df['payload'].str.replace(pattern, r"\1", regex=True)
    df['payload'] = df['payload'].astype(float)
    df['payload2'] = 0
    df['payload2'] = df['payload'].str.replace(pattern, r"\2", regex=True)
    df['payload2'] = df['payload2'].astype(float)
elif function_name == 'workbench-chameleon':
    pattern = r"\{'num_of_rows': (\d+), 'num_of_cols': (\d+)\}"
    df['payload'] = df['payload'].str.replace(pattern, r"\1", regex=True)
    df['payload'] = df['payload'].astype(float)
    df['payload2'] = 0
    df['payload2'] = df['payload'].str.replace(pattern, r"\2", regex=True)
    df['payload2'] = df['payload2'].astype(float)
else:
    # Define the regex pattern
    pattern = r"\{'n': (\d+)\}"
    # Replace the column value with the matched group
    df['payload'] = df['payload'].str.replace(pattern, r"\1", regex=True)
    df['payload'] = df['payload'].astype(float)

df

,payload,request_id,memory_used,start_time,init_duration,memory_size,duration
0,4510.0,8ecbbd9c-be4c-4421-933b-d8f0b5651776,371,1723701853289,None,1792,2343
1,2510.0,36c9b339-5113-4844-a31f-b987a433532e,151,1723701830678,186,896,2180
2,6210.0,826c0a9f-dbdf-4cc0-98b2-e820d3d2d02e,254,1723701846820,None,256,11198
3,6010.0,af76168d-125c-41fa-8d05-27f394c89e69,339,1723701859909,None,1920,3346
4,10.0,c4bed169-700b-41d5-8813-c056e42fe625,43,1723701829129,184,2944,6
...,...,...,...,...,...,...,...
2474,4410.0,20464adb-e251-4d8c-b39f-898f80589f65,328,1723701850806,None,896,4744
2475,4710.0,8208f5db-dcd2-42bc-8235-c14805c6f09f,269,1723701833832,189,896,5336
2476,4810.0,5654acae-d026-40b7-8ebf-5d68cea81d51,293,1723701839442,None,2688,2681
2477,6810.0,188faa7b-d179-4b87-9b58-c3412f5987be,391,1723701861833,None,2688,3873


In [13]:
import boto3, json, base64


response = boto3.client('lambda', region_name='ap-southeast-2').invoke(
            FunctionName="sebs-graph-mst",
            Qualifier="RAM256",
            LogType='Tail',
            InvocationType='RequestResponse',
            Payload=json.dumps({'n': 10})
        )
base64.b64decode(response.get('LogResult')).decode('utf-8')

'START RequestId: c23d3b19-cd2e-4461-99ae-76d1c6529eec Version: 47\nEND RequestId: c23d3b19-cd2e-4461-99ae-76d1c6529eec\nREPORT RequestId: c23d3b19-cd2e-4461-99ae-76d1c6529eec\tDuration: 1.80 ms\tBilled Duration: 2 ms\tMemory Size: 256 MB\tMax Memory Used: 43 MB\t\n'